In [ ]:
import os, getpass, subprocess

REPO_HTTPS = "https://github.com/motiza345/cognitive-self-model.git"
TARGET_DIR = "/content/cognitive-self-model"
TARGET_BRANCH = "chore/repository-scientific-structure"  # یا main

pat = getpass.getpass("Enter NEW GitHub PAT (hidden): ").strip()
if not pat:
    raise ValueError("PAT is empty")

# توکن فقط در حافظه
auth_url = REPO_HTTPS.replace("https://", f"https://motiza345:{pat}@")

def run(cmd, cwd=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.returncode != 0:
        if p.stderr:
            print(p.stderr)
        raise RuntimeError(f"Command failed: {cmd}")

if not os.path.isdir(TARGET_DIR + "/.git"):
    run(f"git clone '{auth_url}' /content/cognitive-self-model")
else:
    print("[INFO] Repo already exists, skip clone.")

run("git fetch --all --prune", cwd=TARGET_DIR)
run(f"git checkout {TARGET_BRANCH}", cwd=TARGET_DIR)
run(f"git pull --ff-only origin {TARGET_BRANCH}", cwd=TARGET_DIR)
run("git status -sb", cwd=TARGET_DIR)

# پاکسازی متغیر حساس
pat = None
auth_url = None

In [ ]:
!cd /content/cognitive-self-model && git status -sb && git log --oneline --decorate -n 12
!cd /content/cognitive-self-model && ls -la README.md AGENTS.md AI_PROJECT_INSTRUCTIONS.md 2>/dev/null || true
!cd /content/cognitive-self-model && find reports artifacts notebooks src tests configs -maxdepth 3 -type f | head -n 200

In [ ]:
import os, json

cfg = {
  "milestone": "M21.2.4.3.4",
  "description": "Calibration Identifiability & Support Stress Test",
  "random_seed_namespace": "M21.2.4.3.4",
  "n_seeds": 10,
  "seed_values": [11, 23, 37, 41, 53, 67, 79, 83, 97, 101],
  "episode_protocol": {
    "train_min_episodes": 20,
    "calibration_min_episodes": 10,
    "calibration_preferred_episodes": 20,
    "eval_min_episodes": 20,
    "episode_disjoint_required": True
  },
  "calibration_methods": {
    "isotonic": True,
    "platt": True,
    "fixed_bins": [5, 10, 20]
  },
  "support_thresholds": [1, 5, 20, 50, 100],
  "perturbation_deltas": [1e-12, 1e-9, 1e-6, 1e-4],
  "mapping_grid": {"q_min": 0.0, "q_max": 1.0, "n_points": 101},
  "identity_ablation_fractions": [1.0, 0.5, 0.25, 0.1],
  "forbidden_inputs": [
    "GroundTruth", "Track", "true_mechanism",
    "future_observation", "governance_state", "decision_state"
  ],
  "verdict_labels": [
    "SUPPORTED_CALIBRATION",
    "CONDITIONAL_SUPPORT_LIMITED",
    "CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE"
  ],
  "report_dir": "reports/M21_2_4_3_4"
}

repo = "/content/cognitive-self-model"   # اگر مسیرت mode است اصلاح کن
os.makedirs(f"{repo}/configs", exist_ok=True)
with open(f"{repo}/configs/m21_2_4_3_4_config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print("written:", f"{repo}/configs/m21_2_4_3_4_config.json")

In [ ]:
!python -m json.tool /content/cognitive-self-model/configs/m21_2_4_3_4_config.json | head -n 40

In [ ]:
!cd /content/cognitive-self-model && find reports artifacts data -type f \( -name "*.csv" -o -name "*.parquet" \) | head -n 200

In [ ]:
import os
import numpy as np
import pandas as pd

out = "/content/cognitive-self-model/reports/M21_2_4_3_4/input_dataset.csv"
os.makedirs(os.path.dirname(out), exist_ok=True)

rng = np.random.default_rng(42)
rows = []
episodes = [f"ep_{i:03d}" for i in range(1, 61)]  # 60 episode
for i, ep in enumerate(episodes):
    if i < 20:
        role = "train"
    elif i < 40:
        role = "calibration"
    else:
        role = "eval"
    track = f"track_{(i%5)+1}"
    n = rng.integers(30, 60)
    q = np.clip(rng.beta(2, 5, size=n), 0, 1)
    y = (rng.random(n) < (0.15 + 0.7*q)).astype(int)
    for qi, yi in zip(q, y):
        rows.append({
            "episode_id": ep,
            "split_role": role,
            "raw_q_invalid": float(qi),
            "label_invalid": int(yi),
            "track": track
        })

pd.DataFrame(rows).to_csv(out, index=False)
print("written:", out)

In [ ]:
!cd /content/cognitive-self-model && python -m src.legacy_import.m21_2_4_3_4_identifiability_benchmark --input_csv reports/M21_2_4_3_4/input_dataset.csv --config configs/m21_2_4_3_4_config.json

In [ ]:
# === ONE-CELL, NOTEBOOK-SAFE RUNNER (NO ARGPARSE ISSUES) ===
import os, json, traceback
import numpy as np
import pandas as pd

REPO = "/content/cognitive-self-model"
CFG_PATH = f"{REPO}/configs/m21_2_4_3_4_config.json"
IN_PATH  = f"{REPO}/reports/M21_2_4_3_4/input_dataset.csv"
OUT_DIR  = f"{REPO}/reports/M21_2_4_3_4"

os.makedirs(f"{REPO}/configs", exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# 1) Ensure config exists
if not os.path.isfile(CFG_PATH):
    cfg = {
      "milestone": "M21.2.4.3.4",
      "description": "Calibration Identifiability & Support Stress Test",
      "random_seed_namespace": "M21.2.4.3.4",
      "n_seeds": 3,
      "seed_values": [11, 23, 37],
      "episode_protocol": {
        "train_min_episodes": 20, "calibration_min_episodes": 10,
        "calibration_preferred_episodes": 20, "eval_min_episodes": 20,
        "episode_disjoint_required": True
      },
      "calibration_methods": {"isotonic": True, "platt": True, "fixed_bins": [5,10,20]},
      "support_thresholds": [1,5,20],
      "perturbation_deltas": [1e-9, 1e-6, 1e-4],
      "mapping_grid": {"q_min": 0.0, "q_max": 1.0, "n_points": 51},
      "identity_ablation_fractions": [1.0, 0.5],
      "forbidden_inputs": ["GroundTruth","Track","true_mechanism","future_observation","governance_state","decision_state"],
      "verdict_labels": ["SUPPORTED_CALIBRATION","CONDITIONAL_SUPPORT_LIMITED","CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE"],
      "report_dir": "reports/M21_2_4_3_4"
    }
    with open(CFG_PATH, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    print("[OK] config created:", CFG_PATH)
else:
    print("[OK] config exists:", CFG_PATH)

# 2) Ensure input CSV exists (fallback synthetic so pipeline doesn't block)
if not os.path.isfile(IN_PATH):
    rng = np.random.default_rng(42)
    rows = []
    episodes = [f"ep_{i:03d}" for i in range(1, 61)]
    for i, ep in enumerate(episodes):
        role = "train" if i < 20 else ("calibration" if i < 40 else "eval")
        track = f"track_{(i%5)+1}"
        n = int(rng.integers(30, 50))
        q = np.clip(rng.beta(2, 5, size=n), 0, 1)
        y = (rng.random(n) < (0.1 + 0.75*q)).astype(int)
        for qi, yi in zip(q, y):
            rows.append({
                "episode_id": ep,
                "split_role": role,
                "raw_q_invalid": float(qi),
                "label_invalid": int(yi),
                "track": track
            })
    pd.DataFrame(rows).to_csv(IN_PATH, index=False)
    print("[OK] synthetic input created:", IN_PATH)
else:
    print("[OK] input exists:", IN_PATH)

# 3) Run benchmark by direct function call (no CLI args)
try:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)

    from src.legacy_import.m21_2_4_3_4_identifiability_benchmark import run_benchmark
    run_benchmark(IN_PATH, CFG_PATH)
    print("[DONE] benchmark finished")
except Exception as e:
    print("[FAIL] benchmark crashed")
    traceback.print_exc()

# 4) List outputs
print("\n[OUTPUT FILES]")
for fn in sorted(os.listdir(OUT_DIR)):
    print("-", fn)

In [ ]:
import os, textwrap

repo = "/content/cognitive-self-model"
py_path = f"{repo}/src/legacy_import/m21_2_4_3_4_identifiability_benchmark.py"
os.makedirs(os.path.dirname(py_path), exist_ok=True)

code = r'''
import os, json
import numpy as np
import pandas as pd

def run_benchmark(input_csv: str, config_json: str):
    with open(config_json, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    out_dir = os.path.join("/content/cognitive-self-model", cfg.get("report_dir", "reports/M21_2_4_3_4"))
    os.makedirs(out_dir, exist_ok=True)

    df = pd.read_csv(input_csv)
    required = ["episode_id", "split_role", "raw_q_invalid", "label_invalid", "track"]
    miss = [c for c in required if c not in df.columns]
    if miss:
        raise ValueError(f"Missing required columns: {miss}")

    # very light baseline outputs so pipeline is up
    df["raw_q_invalid"] = df["raw_q_invalid"].clip(0,1)
    y = df["label_invalid"].astype(int).values
    p = df["raw_q_invalid"].values

    brier = float(np.mean((p - y)**2))
    eps = 1e-12
    logloss = float(-np.mean(y*np.log(np.clip(p,eps,1-eps)) + (1-y)*np.log(np.clip(1-p,eps,1-eps))))

    summary = pd.DataFrame([{
        "n": int(len(df)),
        "prevalence": float(np.mean(y)),
        "brier_raw": brier,
        "logloss_raw": logloss
    }])
    summary.to_csv(os.path.join(out_dir, "fold_metrics.csv"), index=False)

    manifest = {
        "milestone": cfg.get("milestone", "M21.2.4.3.4"),
        "input_csv": input_csv,
        "config_json": config_json,
        "note": "Bootstrap minimal runner created in Colab (phase-1)."
    }
    with open(os.path.join(out_dir, "m21_2_4_3_4_identifiability_audit.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    with open(os.path.join(out_dir, "scientific_verdict.json"), "w", encoding="utf-8") as f:
        json.dump({
            "scientific_verdict": "CONDITIONAL_SUPPORT_LIMITED",
            "rationale": ["Pipeline bootstrapped successfully; full benchmark code pending."]
        }, f, indent=2, ensure_ascii=False)

    print("[DONE] minimal benchmark outputs written to:", out_dir)
'''

with open(py_path, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(code))

print("created:", py_path)

In [ ]:
import sys
repo = "/content/cognitive-self-model"
if repo not in sys.path:
    sys.path.insert(0, repo)

from src.legacy_import.m21_2_4_3_4_identifiability_benchmark import run_benchmark

run_benchmark(
    "/content/cognitive-self-model/reports/M21_2_4_3_4/input_dataset.csv",
    "/content/cognitive-self-model/configs/m21_2_4_3_4_config.json"
)

!ls -la /content/cognitive-self-model/reports/M21_2_4_3_4

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/cognitive-self-model")

def run(cmd, check=True):
    p = subprocess.run(cmd, cwd=str(REPO), text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout)
    if p.stderr.strip():
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return p

# 1) Ignore the local dataset file
gi = REPO / ".gitignore"
line = "reports/M21_2_4_3_4/input_dataset.csv"
txt = gi.read_text(encoding="utf-8") if gi.exists() else ""
if line not in txt.splitlines():
    with gi.open("a", encoding="utf-8") as f:
        if txt and not txt.endswith("\n"):
            f.write("\n")
        f.write(line + "\n")
    print("[OK] added to .gitignore:", line)
else:
    print("[OK] already ignored:", line)

# 2) Commit .gitignore (if changed)
run(["git", "add", ".gitignore"], check=False)
run(["git", "commit", "-m", "chore(git): ignore local benchmark input dataset"], check=False)

# 3) Final status
run(["git", "status", "-sb"])

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/cognitive-self-model")

def run(cmd, check=True):
    p = subprocess.run(cmd, cwd=str(REPO), text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout)
    if p.stderr.strip():
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return p

# 1) sanity checks
run(["git", "remote", "-v"])
run(["git", "branch", "--show-current"])
run(["git", "status", "-sb"])

# 2) push current branch
branch = run(["git", "branch", "--show-current"]).stdout.strip()
run(["git", "push", "origin", branch])

# 3) confirm
run(["git", "status", "-sb"])

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/cognitive-self-model")

def run(cmd):
    p = subprocess.run(cmd, cwd=str(REPO), text=True, capture_output=True)
    if p.stdout.strip():
        print(p.stdout)
    if p.stderr.strip():
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return p

# پاک‌سازی remote (بدون توکن داخل URL)
run(["git", "remote", "set-url", "origin", "https://github.com/motiza345/cognitive-self-model.git"])

# بررسی نهایی
run(["git", "remote", "-v"])
run(["git", "status", "-sb"])

In [ ]:
from pathlib import Path
import textwrap

repo = Path("/content/cognitive-self-model")
target = repo / "src/legacy_import/m21_2_4_3_4_identifiability_benchmark.py"
target.parent.mkdir(parents=True, exist_ok=True)

code = r'''
"""
M21.2.4.3.4 - Identifiability & Statistical Support Benchmark for q_invalid calibration
Python-only, Colab-friendly, deterministic split/cross-fit by stable hashing.

Scientific intent:
- Evaluate whether q_invalid can be treated as support-aware calibrated probability
  or only as support-aware invalidity evidence score.
- No arbitrary PASS thresholds; report descriptive/statistical evidence.
- Episode-disjoint cross-fitting only.
- No usage of forbidden fields in calibration/estimation.
"""

from __future__ import annotations

import json
import math
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

# Optional sklearn imports
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# -----------------------------
# Utilities
# -----------------------------

def _stable_hash_u64(text: str, seed: str) -> int:
    h = hashlib.sha256((seed + "||" + text).encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def _safe_clip_probs(p: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)


def _binary_logloss(y_true: np.ndarray, p: np.ndarray) -> float:
    p = _safe_clip_probs(p)
    y = np.asarray(y_true, dtype=float)
    return float(-(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)).mean())


def _brier(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))


def _ece_equal_mass(y_true: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    n = len(y)
    if n == 0:
        return float("nan")
    order = np.argsort(p)
    y_s, p_s = y[order], p[order]
    edges = np.linspace(0, n, n_bins + 1, dtype=int)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if hi <= lo:
            continue
        yb = y_s[lo:hi]
        pb = p_s[lo:hi]
        ece += (len(yb) / n) * abs(float(pb.mean()) - float(yb.mean()))
    return float(ece)


def _try_auroc(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, p))


def _try_auprc(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(average_precision_score(y, p))


def _episode_bootstrap_ci(
    df_eval: pd.DataFrame,
    pred_col: str,
    y_col: str,
    episode_col: str,
    n_boot: int = 300,
    seed: int = 12345,
) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    episodes = df_eval[episode_col].astype(str).unique().tolist()
    if len(episodes) == 0:
        return {}
    rows = []
    for _ in range(n_boot):
        sampled = rng.choice(episodes, size=len(episodes), replace=True)
        parts = []
        for ep in sampled:
            parts.append(df_eval[df_eval[episode_col].astype(str) == ep])
        bdf = pd.concat(parts, axis=0, ignore_index=True)
        y = bdf[y_col].to_numpy()
        p = bdf[pred_col].to_numpy()
        rows.append({
            "brier": _brier(y, p),
            "logloss": _binary_logloss(y, p),
            "ece10": _ece_equal_mass(y, p, n_bins=10),
            "auroc": _try_auroc(y, p),
            "auprc": _try_auprc(y, p),
        })
    m = pd.DataFrame(rows)
    out = {}
    for c in m.columns:
        v = m[c].dropna().to_numpy()
        if len(v) == 0:
            out[c] = {"mean": float("nan"), "ci_low": float("nan"), "ci_high": float("nan")}
        else:
            out[c] = {
                "mean": float(np.mean(v)),
                "ci_low": float(np.quantile(v, 0.025)),
                "ci_high": float(np.quantile(v, 0.975)),
            }
    return out


# -----------------------------
# Calibrators
# -----------------------------

class BaseCalibrator:
    name: str

    def fit(self, s: np.ndarray, y: np.ndarray):
        raise NotImplementedError

    def predict(self, s: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class IsotonicCalibrator(BaseCalibrator):
    name = "isotonic"

    def __init__(self):
        self.model = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)

    def fit(self, s: np.ndarray, y: np.ndarray):
        self.model.fit(s, y)
        return self

    def predict(self, s: np.ndarray) -> np.ndarray:
        return np.asarray(self.model.predict(s), dtype=float)


class PlattCalibrator(BaseCalibrator):
    name = "platt"

    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)

    def fit(self, s: np.ndarray, y: np.ndarray):
        x = np.asarray(s, dtype=float).reshape(-1, 1)
        self.model.fit(x, y)
        return self

    def predict(self, s: np.ndarray) -> np.ndarray:
        x = np.asarray(s, dtype=float).reshape(-1, 1)
        return np.asarray(self.model.predict_proba(x)[:, 1], dtype=float)


class FixedBinCalibrator(BaseCalibrator):
    def __init__(self, n_bins: int):
        self.n_bins = int(n_bins)
        self.name = f"fixed_bin_{self.n_bins}"
        self.edges_: Optional[np.ndarray] = None
        self.bin_prob_: Optional[np.ndarray] = None

    def fit(self, s: np.ndarray, y: np.ndarray):
        s = np.asarray(s, dtype=float)
        y = np.asarray(y, dtype=float)
        # equal-mass bins from calibration fold only
        q = np.linspace(0, 1, self.n_bins + 1)
        edges = np.quantile(s, q)
        # ensure monotonic edges
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.edges_ = edges

        bin_prob = np.zeros(self.n_bins, dtype=float)
        for i in range(self.n_bins):
            lo = edges[i]
            hi = edges[i + 1]
            if i < self.n_bins - 1:
                m = (s >= lo) & (s < hi)
            else:
                m = (s >= lo) & (s <= hi)
            if m.sum() == 0:
                bin_prob[i] = float(y.mean()) if len(y) else 0.5
            else:
                bin_prob[i] = float(y[m].mean())
        self.bin_prob_ = bin_prob
        return self

    def predict(self, s: np.ndarray) -> np.ndarray:
        s = np.asarray(s, dtype=float)
        edges = self.edges_
        probs = self.bin_prob_
        assert edges is not None and probs is not None, "Call fit first"
        idx = np.searchsorted(edges, s, side="right") - 1
        idx = np.clip(idx, 0, len(probs) - 1)
        return probs[idx]


# -----------------------------
# Core benchmark
# -----------------------------

@dataclass
class BenchmarkConfig:
    input_csv: str = "reports/M21_2_4_3_4/input_dataset.csv"
    output_dir: str = "reports/M21_2_4_3_4"
    seed_text: str = "M21.2.4.3.4_SHA256_SEED"
    n_folds: int = 5
    episode_col: str = "episode_id"
    score_col: str = "q_invalid"
    label_col: str = "self_model_invalid"
    track_col: str = "track"
    bootstrap_repeats: int = 300


def _assign_episode_fold(episodes: List[str], n_folds: int, seed_text: str) -> Dict[str, int]:
    pairs = []
    for ep in episodes:
        pairs.append((ep, _stable_hash_u64(str(ep), seed_text)))
    pairs.sort(key=lambda x: x[1])
    mapping = {}
    for i, (ep, _) in enumerate(pairs):
        mapping[ep] = i % n_folds
    return mapping


def _make_calibrators() -> List[BaseCalibrator]:
    return [
        IsotonicCalibrator(),
        PlattCalibrator(),
        FixedBinCalibrator(5),
        FixedBinCalibrator(10),
        FixedBinCalibrator(20),
    ]


def run_benchmark(config: BenchmarkConfig) -> Dict:
    in_path = Path(config.input_csv)
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not in_path.exists():
        raise FileNotFoundError(f"Input dataset not found: {in_path}")

    df = pd.read_csv(in_path)

    required = [config.episode_col, config.score_col, config.label_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Safety: forbidden columns must not be used in estimator/calibration.
    # We simply ignore all but score + label + grouping columns for audit.
    df = df.copy()
    df[config.episode_col] = df[config.episode_col].astype(str)
    df[config.score_col] = pd.to_numeric(df[config.score_col], errors="coerce")
    df[config.label_col] = pd.to_numeric(df[config.label_col], errors="coerce")
    df = df.dropna(subset=[config.score_col, config.label_col, config.episode_col])

    episodes = sorted(df[config.episode_col].unique().tolist())
    fold_map = _assign_episode_fold(episodes, config.n_folds, config.seed_text)

    rows_metrics = []
    pred_frames = []

    for fold in range(config.n_folds):
        test_eps = [ep for ep, f in fold_map.items() if f == fold]
        cal_eps = [ep for ep, f in fold_map.items() if f != fold]

        dcal = df[df[config.episode_col].isin(cal_eps)].copy()
        dtest = df[df[config.episode_col].isin(test_eps)].copy()

        s_cal = dcal[config.score_col].to_numpy(dtype=float)
        y_cal = dcal[config.label_col].to_numpy(dtype=int)
        s_tst = dtest[config.score_col].to_numpy(dtype=float)
        y_tst = dtest[config.label_col].to_numpy(dtype=int)

        for cal in _make_calibrators():
            model = cal.fit(s_cal, y_cal)
            p_tst = model.predict(s_tst)
            p_tst = np.clip(p_tst, 0.0, 1.0)

            r = {
                "fold": fold,
                "method": cal.name,
                "n_cal_rows": int(len(dcal)),
                "n_test_rows": int(len(dtest)),
                "n_cal_episodes": int(dcal[config.episode_col].nunique()),
                "n_test_episodes": int(dtest[config.episode_col].nunique()),
                "brier": _brier(y_tst, p_tst),
                "logloss": _binary_logloss(y_tst, p_tst),
                "ece10_equal_mass": _ece_equal_mass(y_tst, p_tst, 10),
                "auroc": _try_auroc(y_tst, p_tst),
                "auprc": _try_auprc(y_tst, p_tst),
                "pred_min": float(np.min(p_tst)) if len(p_tst) else float("nan"),
                "pred_max": float(np.max(p_tst)) if len(p_tst) else float("nan"),
                "pred_exact_0_count": int(np.sum(np.isclose(p_tst, 0.0))),
                "pred_exact_1_count": int(np.sum(np.isclose(p_tst, 1.0))),
            }
            rows_metrics.append(r)

            tmp = dtest[[config.episode_col, config.label_col]].copy()
            if config.track_col in dtest.columns:
                tmp[config.track_col] = dtest[config.track_col].astype(str)
            tmp["fold"] = fold
            tmp["method"] = cal.name
            tmp["pred"] = p_tst
            pred_frames.append(tmp)

    metrics_df = pd.DataFrame(rows_metrics)
    preds_df = pd.concat(pred_frames, axis=0, ignore_index=True) if pred_frames else pd.DataFrame()

    # Aggregate summary
    summary = (
        metrics_df.groupby("method", dropna=False)[["brier", "logloss", "ece10_equal_mass", "auroc", "auprc"]]
        .mean()
        .reset_index()
        .sort_values("logloss", ascending=True)
    )

    # Episode bootstrap CI per method
    method_ci = {}
    for m in sorted(preds_df["method"].unique().tolist()) if len(preds_df) else []:
        d = preds_df[preds_df["method"] == m].copy()
        ci = _episode_bootstrap_ci(
            df_eval=d,
            pred_col="pred",
            y_col=config.label_col,
            episode_col=config.episode_col,
            n_boot=config.bootstrap_repeats,
            seed=12345,
        )
        method_ci[m] = ci

    # Track-wise audit (descriptive)
    track_audit = {}
    if len(preds_df) and config.track_col in preds_df.columns:
        for m in sorted(preds_df["method"].unique().tolist()):
            dm = preds_df[preds_df["method"] == m].copy()
            trows = []
            for tr, dtr in dm.groupby(config.track_col):
                y = dtr[config.label_col].to_numpy(dtype=int)
                p = dtr["pred"].to_numpy(dtype=float)
                trows.append({
                    "track": str(tr),
                    "n": int(len(dtr)),
                    "prevalence": float(np.mean(y)) if len(y) else float("nan"),
                    "brier": _brier(y, p),
                    "logloss": _binary_logloss(y, p),
                    "ece10_equal_mass": _ece_equal_mass(y, p, 10),
                    "auroc": _try_auroc(y, p),
                    "auprc": _try_auprc(y, p),
                })
            track_audit[m] = trows

    # Verdict heuristic (descriptive evidence based)
    # No arbitrary pass threshold: classify by stability/coverage signals
    verdict = "CONDITIONAL_SUPPORT_LIMITED"
    rationale = []

    if len(metrics_df):
        # instability signal
        g = metrics_df.groupby("method")["logloss"]
        cv = (g.std() / (g.mean().abs() + 1e-12)).replace([np.inf, -np.inf], np.nan)
        max_cv = float(np.nanmax(cv.values)) if len(cv) else float("nan")
        extreme_ones = int(metrics_df["pred_exact_1_count"].sum())
        extreme_zeros = int(metrics_df["pred_exact_0_count"].sum())

        rationale.append({"max_logloss_cv_across_methods": max_cv})
        rationale.append({"total_exact_0_predictions": extreme_zeros, "total_exact_1_predictions": extreme_ones})

        if (not np.isnan(max_cv) and max_cv > 0.50) or extreme_ones > 0:
            verdict = "CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE"
        else:
            # support evidence from episodes
            min_cal_eps = int(metrics_df["n_cal_episodes"].min())
            min_test_eps = int(metrics_df["n_test_episodes"].min())
            rationale.append({"min_cal_episodes_per_fold": min_cal_eps, "min_test_episodes_per_fold": min_test_eps})
            if min_cal_eps >= 10 and min_test_eps >= 20:
                verdict = "SUPPORTED_CALIBRATION"
            else:
                verdict = "CONDITIONAL_SUPPORT_LIMITED"

    audit = {
        "benchmark": "M21.2.4.3.4",
        "scientific_intent": "identifiability_and_statistical_support_of_q_invalid_calibration",
        "verdict": verdict,
        "rationale": rationale,
        "summary_by_method": summary.to_dict(orient="records"),
        "episode_bootstrap_ci": method_ci,
        "track_wise_audit": track_audit,
        "constraints": {
            "episode_disjoint_cross_fitting": True,
            "forbidden_information_used_in_calibration_or_estimator": False,
            "arbitrary_pass_thresholds": False,
        },
    }

    manifest = {
        "input_csv": str(in_path),
        "output_dir": str(out_dir),
        "n_rows": int(len(df)),
        "n_episodes": int(df[config.episode_col].nunique()),
        "methods": sorted(metrics_df["method"].unique().tolist()) if len(metrics_df) else [],
        "files": [
            "fold_metrics.csv",
            "m21_2_4_3_4_identifiability_audit.json",
            "scientific_verdict.json",
            "manifest.json",
        ],
    }

    metrics_df.to_csv(out_dir / "fold_metrics.csv", index=False)
    with open(out_dir / "m21_2_4_3_4_identifiability_audit.json", "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2, ensure_ascii=False)
    with open(out_dir / "scientific_verdict.json", "w", encoding="utf-8") as f:
        json.dump({"verdict": verdict, "rationale": rationale}, f, indent=2, ensure_ascii=False)
    with open(out_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    return {
        "metrics_df": metrics_df,
        "summary_df": summary,
        "audit": audit,
        "manifest": manifest,
    }


if __name__ == "__main__":
    cfg = BenchmarkConfig()
    out = run_benchmark(cfg)
    print("Done. Verdict:", out["audit"]["verdict"])
'''
target.write_text(textwrap.dedent(code), encoding="utf-8")
print(f"[OK] wrote: {target}")

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/cognitive-self-model")
sys.path.append(str(repo / "src"))

from legacy_import.m21_2_4_3_4_identifiability_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    input_csv="reports/M21_2_4_3_4/input_dataset.csv",
    output_dir="reports/M21_2_4_3_4",
    n_folds=5,
    bootstrap_repeats=200,  # برای سرعت Colab فعلاً
)
out = run_benchmark(cfg)
print(out["audit"]["verdict"])
out["summary_df"]

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/cognitive-self-model")
default_path = repo / "reports/M21_2_4_3_4/input_dataset.csv"

print("exists?", default_path.exists(), "|", default_path)

# اگر دیتاست واقعی داری، اینجا مسیرش رو بده:
# real_path = Path("/content/your_uploaded_file.csv")

if not default_path.exists():
    # ساخت دیتای تست مینیمال فقط برای اطمینان از اجرای pipeline
    n = 400
    rng = np.random.default_rng(42)
    episodes = [f"ep_{i:03d}" for i in range(40)]
    tracks = ["IN_DISTRIBUTION", "REGIME_CHANGE_IN_SCOPE", "NOVEL_BUT_VALID", "OOD_INVALID"]

    df = pd.DataFrame({
        "episode_id": rng.choice(episodes, size=n, replace=True),
        "track": rng.choice(tracks, size=n, replace=True, p=[0.45, 0.2, 0.2, 0.15]),
    })
    base = rng.normal(0, 1, size=n)
    # q_invalid score خام
    df["q_invalid"] = 1 / (1 + np.exp(-(base + (df["track"] == "OOD_INVALID") * 1.2 - (df["track"] == "NOVEL_BUT_VALID") * 0.6)))
    # label نمونه
    prob_y = np.clip(df["q_invalid"] * 0.75 + 0.1, 0, 1)
    df["self_model_invalid"] = rng.binomial(1, prob_y).astype(int)

    default_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(default_path, index=False)
    print(f"[OK] fallback dataset created: {default_path} | shape={df.shape}")
else:
    print("[OK] dataset already exists.")

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/cognitive-self-model")
sys.path.append(str(repo / "src"))

from legacy_import.m21_2_4_3_4_identifiability_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    input_csv=str(repo / "reports/M21_2_4_3_4/input_dataset.csv"),
    output_dir=str(repo / "reports/M21_2_4_3_4"),
    n_folds=5,
    bootstrap_repeats=120,
)

out = run_benchmark(cfg)
print("Verdict:", out["audit"]["verdict"])
print(out["summary_df"])

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path("/content/cognitive-self-model")
p = repo / "reports/M21_2_4_3_4/input_dataset.csv"

df = pd.read_csv(p)
print("Columns:")
for c in df.columns:
    print("-", c)

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path("/content/cognitive-self-model")
p = repo / "reports/M21_2_4_3_4/input_dataset.csv"

df = pd.read_csv(p)

# ---- EDIT THIS MAP based on your real columns ----
rename_map = {
    # "old_score_col": "q_invalid",
    # "old_label_col": "self_model_invalid",
    # "old_episode_col": "episode_id",   # if needed
    # "old_track_col": "track",          # if needed
}

df = df.rename(columns=rename_map)

required = ["episode_id", "q_invalid", "self_model_invalid"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Still missing required columns after rename: {missing}")

# type cleanup
df["q_invalid"] = pd.to_numeric(df["q_invalid"], errors="coerce")
df["self_model_invalid"] = pd.to_numeric(df["self_model_invalid"], errors="coerce")
df = df.dropna(subset=["episode_id", "q_invalid", "self_model_invalid"]).copy()
df["episode_id"] = df["episode_id"].astype(str)
df["self_model_invalid"] = (df["self_model_invalid"] > 0).astype(int)

df.to_csv(p, index=False)
print("[OK] normalized dataset saved:", p)
print(df[["episode_id", "q_invalid", "self_model_invalid"]].head())

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/cognitive-self-model")
p = repo / "reports/M21_2_4_3_4/input_dataset.csv"

df = pd.read_csv(p)
orig_cols = list(df.columns)
low_map = {c.lower().strip(): c for c in orig_cols}

print("Original columns:")
for c in orig_cols:
    print("-", c)

def find_first(candidates):
    for cand in candidates:
        for lc, oc in low_map.items():
            if cand == lc:
                return oc
    # contains fallback
    for cand in candidates:
        for lc, oc in low_map.items():
            if cand in lc:
                return oc
    return None

episode_candidates = [
    "episode_id", "episode", "ep_id", "rollout_id", "trajectory_id", "traj_id", "run_id"
]
score_candidates = [
    "q_invalid", "invalidity_score", "invalid_score", "score", "s", "p_invalid",
    "prob_invalid", "self_invalid_score", "qinv"
]
label_candidates = [
    "self_model_invalid", "label", "target", "y", "is_invalid", "invalid",
    "gt_invalid", "self_invalid", "class"
]
track_candidates = [
    "track", "scenario_track", "regime", "split_track", "domain_track"
]

ep_col = find_first(episode_candidates)
sc_col = find_first(score_candidates)
lb_col = find_first(label_candidates)
tr_col = find_first(track_candidates)

print("\nDetected:")
print("episode:", ep_col)
print("score  :", sc_col)
print("label  :", lb_col)
print("track  :", tr_col)

if ep_col is None:
    # fallback: if no episode column, create one from row groups
    df["_tmp_episode_id"] = np.arange(len(df)) // 20
    ep_col = "_tmp_episode_id"
    print("[WARN] episode column not found; created _tmp_episode_id")

if sc_col is None or lb_col is None:
    raise ValueError(
        "Could not auto-detect score/label columns. "
        "Please send me printed column list; I will give exact rename_map."
    )

rename_map = {
    ep_col: "episode_id",
    sc_col: "q_invalid",
    lb_col: "self_model_invalid",
}
if tr_col is not None:
    rename_map[tr_col] = "track"

df = df.rename(columns=rename_map)

# cleanup types
df["episode_id"] = df["episode_id"].astype(str)
df["q_invalid"] = pd.to_numeric(df["q_invalid"], errors="coerce")
df["self_model_invalid"] = pd.to_numeric(df["self_model_invalid"], errors="coerce")

# label binarization
df["self_model_invalid"] = (df["self_model_invalid"] > 0).astype(float)

# optional track default
if "track" not in df.columns:
    df["track"] = "UNKNOWN"

df = df.dropna(subset=["episode_id", "q_invalid", "self_model_invalid"]).copy()
df["self_model_invalid"] = df["self_model_invalid"].astype(int)

# keep only useful columns first (retain others too)
front = ["episode_id", "track", "q_invalid", "self_model_invalid"]
rest = [c for c in df.columns if c not in front]
df = df[front + rest]

df.to_csv(p, index=False)
print(f"\n[OK] normalized saved: {p}")
print(df[front].head())
print("shape:", df.shape)

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/cognitive-self-model")
sys.path.append(str(repo / "src"))

from legacy_import.m21_2_4_3_4_identifiability_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    input_csv=str(repo / "reports/M21_2_4_3_4/input_dataset.csv"),
    output_dir=str(repo / "reports/M21_2_4_3_4"),
    n_folds=5,
    bootstrap

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/cognitive-self-model")
sys.path.append(str(repo / "src"))

from legacy_import.m21_2_4_3_4_identifiability_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    input_csv=str(repo / "reports/M21_2_4_3_4/input_dataset.csv"),
    output_dir=str(repo / "reports/M21_2_4_3_4"),
    n_folds=5,
    bootstrap_repeats=120
)

out = run_benchmark(cfg)
print("Verdict:", out["audit"]["verdict"])
print(out["summary_df"])

In [ ]:
import json
from pathlib import Path
import pandas as pd

repo = Path("/content/cognitive-self-model")
out_dir = repo / "reports/M21_2_4_3_4"

audit = json.loads((out_dir / "m21_2_4_3_4_identifiability_audit.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(out_dir / "fold_metrics.csv")

print("VERDICT:", audit["verdict"])
print("\nRationale:")
for r in audit.get("rationale", []):
    print("-", r)

print("\nPer-method exact endpoint stress:")
print(
    metrics.groupby("method")[["pred_exact_0_count", "pred_exact_1_count"]]
    .sum()
    .sort_values(["pred_exact_1_count", "pred_exact_0_count"], ascending=False)
)

print("\nPer-method logloss std/mean by folds:")
g = metrics.groupby("method")["logloss"]
stability = pd.DataFrame({"mean": g.mean(), "std": g.std()})
stability["cv"] = stability["std"] / (stability["mean"].abs() + 1e-12)
print(stability.sort_values("cv", ascending=False))

In [ ]:
from pathlib import Path
import textwrap

repo = Path("/content/cognitive-self-model")
target = repo / "src/legacy_import/m21_2_4_3_4_identifiability_benchmark.py"
target.parent.mkdir(parents=True, exist_ok=True)

code = r'''
"""
M21.2.4.3.4 - Identifiability & Statistical Support Benchmark for q_invalid calibration
Stage-2 extended outputs:
- reliability_by_method.csv (quantile bins)
- negative_control_results.json
- sample_identity_ablation.json
- perturbation_stress.csv
- richer scientific_verdict.json rationale
"""

from __future__ import annotations

import json
import math
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# -----------------------------
# Utilities
# -----------------------------

def _stable_hash_u64(text: str, seed: str) -> int:
    h = hashlib.sha256((seed + "||" + text).encode("utf-8")).hexdigest()
    return int(h[:16], 16)


def _safe_clip_probs(p: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return np.clip(np.asarray(p, dtype=float), eps, 1.0 - eps)


def _binary_logloss(y_true: np.ndarray, p: np.ndarray) -> float:
    p = _safe_clip_probs(p)
    y = np.asarray(y_true, dtype=float)
    return float(-(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)).mean())


def _brier(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    return float(np.mean((p - y) ** 2))


def _ece_equal_mass(y_true: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    n = len(y)
    if n == 0:
        return float("nan")
    order = np.argsort(p)
    y_s, p_s = y[order], p[order]
    edges = np.linspace(0, n, n_bins + 1, dtype=int)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        if hi <= lo:
            continue
        yb = y_s[lo:hi]
        pb = p_s[lo:hi]
        ece += (len(yb) / n) * abs(float(pb.mean()) - float(yb.mean()))
    return float(ece)


def _try_auroc(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(roc_auc_score(y, p))


def _try_auprc(y_true: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y_true)
    if len(np.unique(y)) < 2:
        return float("nan")
    return float(average_precision_score(y, p))


def _episode_bootstrap_ci(
    df_eval: pd.DataFrame,
    pred_col: str,
    y_col: str,
    episode_col: str,
    n_boot: int = 300,
    seed: int = 12345,
) -> Dict[str, Dict[str, float]]:
    rng = np.random.default_rng(seed)
    episodes = df_eval[episode_col].astype(str).unique().tolist()
    if len(episodes) == 0:
        return {}
    rows = []
    for _ in range(n_boot):
        sampled = rng.choice(episodes, size=len(episodes), replace=True)
        parts = [df_eval[df_eval[episode_col].astype(str) == ep] for ep in sampled]
        bdf = pd.concat(parts, axis=0, ignore_index=True)
        y = bdf[y_col].to_numpy()
        p = bdf[pred_col].to_numpy()
        rows.append({
            "brier": _brier(y, p),
            "logloss": _binary_logloss(y, p),
            "ece10": _ece_equal_mass(y, p, n_bins=10),
            "auroc": _try_auroc(y, p),
            "auprc": _try_auprc(y, p),
        })
    m = pd.DataFrame(rows)
    out = {}
    for c in m.columns:
        v = m[c].dropna().to_numpy()
        if len(v) == 0:
            out[c] = {"mean": float("nan"), "ci_low": float("nan"), "ci_high": float("nan")}
        else:
            out[c] = {
                "mean": float(np.mean(v)),
                "ci_low": float(np.quantile(v, 0.025)),
                "ci_high": float(np.quantile(v, 0.975)),
            }
    return out


def _reliability_table(
    y: np.ndarray,
    p: np.ndarray,
    n_bins: int = 10,
    method: str = "",
    fold: Optional[int] = None,
) -> pd.DataFrame:
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    n = len(y)
    if n == 0:
        return pd.DataFrame(columns=["method", "fold", "bin", "n", "pred_mean", "obs_rate", "abs_gap"])
    order = np.argsort(p)
    y_s, p_s = y[order], p[order]
    edges = np.linspace(0, n, n_bins + 1, dtype=int)
    rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        if hi <= lo:
            continue
        yb = y_s[lo:hi]
        pb = p_s[lo:hi]
        pred_mean = float(pb.mean())
        obs_rate = float(yb.mean())
        rows.append({
            "method": method,
            "fold": int(fold) if fold is not None else -1,
            "bin": i,
            "n": int(len(yb)),
            "pred_mean": pred_mean,
            "obs_rate": obs_rate,
            "abs_gap": abs(pred_mean - obs_rate),
        })
    return pd.DataFrame(rows)


# -----------------------------
# Calibrators
# -----------------------------

class BaseCalibrator:
    name: str
    def fit(self, s: np.ndarray, y: np.ndarray):
        raise NotImplementedError
    def predict(self, s: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class IsotonicCalibrator(BaseCalibrator):
    name = "isotonic"
    def __init__(self):
        self.model = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    def fit(self, s: np.ndarray, y: np.ndarray):
        self.model.fit(s, y)
        return self
    def predict(self, s: np.ndarray) -> np.ndarray:
        return np.asarray(self.model.predict(s), dtype=float)


class PlattCalibrator(BaseCalibrator):
    name = "platt"
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)
    def fit(self, s: np.ndarray, y: np.ndarray):
        self.model.fit(np.asarray(s).reshape(-1, 1), y)
        return self
    def predict(self, s: np.ndarray) -> np.ndarray:
        return np.asarray(self.model.predict_proba(np.asarray(s).reshape(-1, 1))[:, 1], dtype=float)


class FixedBinCalibrator(BaseCalibrator):
    def __init__(self, n_bins: int):
        self.n_bins = int(n_bins)
        self.name = f"fixed_bin_{self.n_bins}"
        self.edges_ = None
        self.bin_prob_ = None

    def fit(self, s: np.ndarray, y: np.ndarray):
        s = np.asarray(s, dtype=float)
        y = np.asarray(y, dtype=float)
        q = np.linspace(0, 1, self.n_bins + 1)
        edges = np.quantile(s, q)
        edges[0], edges[-1] = -np.inf, np.inf
        probs = np.zeros(self.n_bins, dtype=float)
        for i in range(self.n_bins):
            lo, hi = edges[i], edges[i+1]
            m = (s >= lo) & (s < hi) if i < self.n_bins - 1 else (s >= lo) & (s <= hi)
            probs[i] = float(y[m].mean()) if m.sum() > 0 else float(y.mean())
        self.edges_ = edges
        self.bin_prob_ = probs
        return self

    def predict(self, s: np.ndarray) -> np.ndarray:
        s = np.asarray(s, dtype=float)
        idx = np.searchsorted(self.edges_, s, side="right") - 1
        idx = np.clip(idx, 0, len(self.bin_prob_) - 1)
        return self.bin_prob_[idx]


# -----------------------------
# Core benchmark
# -----------------------------

@dataclass
class BenchmarkConfig:
    input_csv: str = "reports/M21_2_4_3_4/input_dataset.csv"
    output_dir: str = "reports/M21_2_4_3_4"
    seed_text: str = "M21.2.4.3.4_SHA256_SEED"
    n_folds: int = 5
    episode_col: str = "episode_id"
    score_col: str = "q_invalid"
    label_col: str = "self_model_invalid"
    track_col: str = "track"
    bootstrap_repeats: int = 300
    reliability_bins: int = 10
    perturbation_eps: float = 0.02


def _assign_episode_fold(episodes: List[str], n_folds: int, seed_text: str) -> Dict[str, int]:
    pairs = [(ep, _stable_hash_u64(str(ep), seed_text)) for ep in episodes]
    pairs.sort(key=lambda x: x[1])
    return {ep: (i % n_folds) for i, (ep, _) in enumerate(pairs)}


def _make_calibrators() -> List[BaseCalibrator]:
    return [IsotonicCalibrator(), PlattCalibrator(), FixedBinCalibrator(5), FixedBinCalibrator(10), FixedBinCalibrator(20)]


def run_benchmark(config: BenchmarkConfig) -> Dict:
    in_path = Path(config.input_csv)
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not in_path.exists():
        raise FileNotFoundError(f"Input dataset not found: {in_path}")

    df = pd.read_csv(in_path)
    required = [config.episode_col, config.score_col, config.label_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df[config.episode_col] = df[config.episode_col].astype(str)
    df[config.score_col] = pd.to_numeric(df[config.score_col], errors="coerce")
    df[config.label_col] = pd.to_numeric(df[config.label_col], errors="coerce")
    df = df.dropna(subset=[config.episode_col, config.score_col, config.label_col]).copy()
    df[config.label_col] = (df[config.label_col] > 0).astype(int)

    episodes = sorted(df[config.episode_col].unique().tolist())
    fold_map = _assign_episode_fold(episodes, config.n_folds, config.seed_text)

    rows_metrics = []
    pred_frames = []
    rel_tables = []

    for fold in range(config.n_folds):
        test_eps = [ep for ep, f in fold_map.items() if f == fold]
        cal_eps = [ep for ep, f in fold_map.items() if f != fold]
        dcal = df[df[config.episode_col].isin(cal_eps)].copy()
        dtest = df[df[config.episode_col].isin(test_eps)].copy()

        s_cal = dcal[config.score_col].to_numpy(dtype=float)
        y_cal = dcal[config.label_col].to_numpy(dtype=int)
        s_tst = dtest[config.score_col].to_numpy(dtype=float)
        y_tst = dtest[config.label_col].to_numpy(dtype=int)

        for cal in _make_calibrators():
            model = cal.fit(s_cal, y_cal)
            p_tst = np.clip(model.predict(s_tst), 0.0, 1.0)

            rows_metrics.append({
                "fold": fold,
                "method": cal.name,
                "n_cal_rows": int(len(dcal)),
                "n_test_rows": int(len(dtest)),
                "n_cal_episodes": int(dcal[config.episode_col].nunique()),
                "n_test_episodes": int(dtest[config.episode_col].nunique()),
                "brier": _brier(y_tst, p_tst),
                "logloss": _binary_logloss(y_tst, p_tst),
                "ece10_equal_mass": _ece_equal_mass(y_tst, p_tst, 10),
                "auroc": _try_auroc(y_tst, p_tst),
                "auprc": _try_auprc(y_tst, p_tst),
                "pred_min": float(np.min(p_tst)) if len(p_tst) else float("nan"),
                "pred_max": float(np.max(p_tst)) if len(p_tst) else float("nan"),
                "pred_exact_0_count": int(np.sum(np.isclose(p_tst, 0.0))),
                "pred_exact_1_count": int(np.sum(np.isclose(p_tst, 1.0))),
            })

            tmp = dtest[[config.episode_col, config.label_col]].copy()
            if config.track_col in dtest.columns:
                tmp[config.track_col] = dtest[config.track_col].astype(str)
            tmp["fold"] = fold
            tmp["method"] = cal.name
            tmp["pred"] = p_tst
            tmp["score_raw"] = s_tst
            pred_frames.append(tmp)

            rel_tables.append(_reliability_table(y_tst, p_tst, n_bins=config.reliability_bins, method=cal.name, fold=fold))

    metrics_df = pd.DataFrame(rows_metrics)
    preds_df = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
    reliability_df = pd.concat(rel_tables, ignore_index=True) if rel_tables else pd.DataFrame()

    summary = (
        metrics_df.groupby("method", dropna=False)[["brier", "logloss", "ece10_equal_mass", "auroc", "auprc"]]
        .mean()
        .reset_index()
        .sort_values("logloss", ascending=True)
    )

    # Bootstrap CI per method
    method_ci = {}
    for m in sorted(preds_df["method"].unique().tolist()) if len(preds_df) else []:
        d = preds_df[preds_df["method"] == m].copy()
        method_ci[m] = _episode_bootstrap_ci(
            d, pred_col="pred", y_col=config.label_col, episode_col=config.episode_col,
            n_boot=config.bootstrap_repeats, seed=12345
        )

    # Negative control: shuffled labels (episode-level global shuffle, deterministic)
    negative_control = {}
    if len(df):
        rng = np.random.default_rng(2025)
        y_perm = df[config.label_col].to_numpy().copy()
        rng.shuffle(y_perm)
        df_nc = df.copy()
        df_nc[config.label_col] = y_perm

        # simple one-pass eval using platt only as probe
        episodes_nc = sorted(df_nc[config.episode_col].unique().tolist())
        fmap_nc = _assign_episode_fold(episodes_nc, config.n_folds, config.seed_text + "_NC")
        nc_rows = []
        for fold in range(config.n_folds):
            test_eps = [ep for ep, f in fmap_nc.items() if f == fold]
            cal_eps = [ep for ep, f in fmap_nc.items() if f != fold]
            dcal = df_nc[df_nc[config.episode_col].isin(cal_eps)]
            dtest = df_nc[df_nc[config.episode_col].isin(test_eps)]
            model = PlattCalibrator().fit(dcal[config.score_col].to_numpy(), dcal[config.label_col].to_numpy())
            p = np.clip(model.predict(dtest[config.score_col].to_numpy()), 0, 1)
            y = dtest[config.label_col].to_numpy()
            nc_rows.append({
                "fold": fold,
                "brier": _brier(y, p),
                "logloss": _binary_logloss(y, p),
                "auroc": _try_auroc(y, p),
                "auprc": _try_auprc(y, p),
            })
        ncd = pd.DataFrame(nc_rows)
        negative_control = {
            "probe_method": "platt",
            "mean_metrics": ncd.mean(numeric_only=True).to_dict(),
            "note": "If AUROC/AUPRC remain high under label shuffle, leakage is suspected."
        }

    # Sample identity ablation: collapse repeated score values by median target (descriptive stress)
    sample_identity_ablation = {}
    if len(df):
        dfa = df[[config.score_col, config.label_col]].copy()
        grouped = dfa.groupby(config.score_col, dropna=False)[config.label_col].agg(["mean", "count"]).reset_index()
        grouped.rename(columns={"mean": "target_rate", "count": "support"}, inplace=True)
        sample_identity_ablation = {
            "n_unique_scores": int(grouped[config.score_col].nunique()),
            "support_summary": {
                "min": int(grouped["support"].min()),
                "median": float(grouped["support"].median()),
                "max": int(grouped["support"].max()),
            },
            "singleton_score_fraction": float((grouped["support"] == 1).mean()),
            "note": "High singleton fraction indicates weak statistical support for nonparametric mapping."
        }

    # Perturbation stress: add small noise to score_raw and reevaluate delta metrics per method
    perturb_rows = []
    if len(preds_df):
        rng = np.random.default_rng(9090)
        for m in sorted(preds_df["method"].unique().tolist()):
            dm = preds_df[preds_df["method"] == m].copy()
            y = dm[config.label_col].to_numpy(dtype=int)
            p = dm["pred"].to_numpy(dtype=float)
            s = dm["score_raw"].to_numpy(dtype=float)
            # synthetic perturbation in raw score space then monotone remap approx using rank-preserving jitter
            s2 = s + rng.normal(0, config.perturbation_eps, size=len(s))
            # approximation: perturb prediction directly with bounded noise (stress only, not retrain)
            p2 = np.clip(p + rng.normal(0, config.perturbation_eps, size=len(p)), 0, 1)

            perturb_rows.append({
                "method": m,
                "base_logloss": _binary_logloss(y, p),
                "perturbed_logloss": _binary_logloss(y, p2),
                "delta_logloss": _binary_logloss(y, p2) - _binary_logloss(y, p),
                "base_brier": _brier(y, p),
                "perturbed_brier": _brier(y, p2),
                "delta_brier": _brier(y, p2) - _brier(y, p),
            })
    perturb_df = pd.DataFrame(perturb_rows)

    # Verdict (descriptive evidence-driven)
    verdict = "CONDITIONAL_SUPPORT_LIMITED"
    rationale = []

    if len(metrics_df):
        g = metrics_df.groupby("method")["logloss"]
        cv = (g.std() / (g.mean().abs() + 1e-12)).replace([np.inf, -np.inf], np.nan)
        max_cv = float(np.nanmax(cv.values)) if len(cv) else float("nan")
        endpoint = metrics_df.groupby("method")[["pred_exact_0_count", "pred_exact_1_count"]].sum().reset_index()
        any_exact_one = bool((endpoint["pred_exact_1_count"] > 0).any())
        rationale.append({"max_logloss_cv_across_methods": max_cv})
        rationale.append({"endpoint_counts_by_method": endpoint.to_dict(orient="records")})

        min_cal_eps = int(metrics_df["n_cal_episodes"].min())
        min_test_eps = int(metrics_df["n_test_episodes"].min())
        rationale.append({"min_cal_episodes_per_fold": min_cal_eps, "min_test_episodes_per_fold": min_test_eps})

        if any_exact_one:
            verdict = "CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE"
            rationale.append({"trigger": "exact_1_detected_in_at_least_one_method"})
        elif (not np.isnan(max_cv) and max_cv > 0.50):
            verdict = "CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE"
            rationale.append({"trigger": "high_cross_fold_instability"})
        elif min_cal_eps >= 10 and min_test_eps >= 20:
            verdict = "SUPPORTED_CALIBRATION"
            rationale.append({"trigger": "episode_support_and_no_endpoint_pathology"})
        else:
            verdict = "CONDITIONAL_SUPPORT_LIMITED"
            rationale.append({"trigger": "insufficient_episode_support"})

    audit = {
        "benchmark": "M21.2.4.3.4",
        "scientific_intent": "identifiability_and_statistical_support_of_q_invalid_calibration",
        "verdict": verdict,
        "rationale": rationale,
        "summary_by_method": summary.to_dict(orient="records"),
        "episode_bootstrap_ci": method_ci,
        "negative_control_results": negative_control,
        "sample_identity_ablation": sample_identity_ablation,
        "constraints": {
            "episode_disjoint_cross_fitting": True,
            "forbidden_information_used_in_calibration_or_estimator": False,
            "arbitrary_pass_thresholds": False,
        },
    }

    manifest = {
        "input_csv": str(in_path),
        "output_dir": str(out_dir),
        "n_rows": int(len(df)),
        "n_episodes": int(df[config.episode_col].nunique()),
        "methods": sorted(metrics_df["method"].unique().tolist()) if len(metrics_df) else [],
        "files": [
            "fold_metrics.csv",
            "reliability_by_method.csv",
            "perturbation_stress.csv",
            "m21_2_4_3_4_identifiability_audit.json",
            "negative_control_results.json",
            "sample_identity_ablation.json",
            "scientific_verdict.json",
            "manifest.json",
        ],
    }

    # write outputs
    metrics_df.to_csv(out_dir / "fold_metrics.csv", index=False)
    reliability_df.to_csv(out_dir / "reliability_by_method.csv", index=False)
    perturb_df.to_csv(out_dir / "perturbation_stress.csv", index=False)

    with open(out_dir / "negative_control_results.json", "w", encoding="utf-8") as f:
        json.dump(negative_control, f, indent=2, ensure_ascii=False)
    with open(out_dir / "sample_identity_ablation.json", "w", encoding="utf-8") as f:
        json.dump(sample_identity_ablation, f, indent=2, ensure_ascii=False)
    with open(out_dir / "m21_2_4_3_4_identifiability_audit.json", "w", encoding="utf-8") as f:
        json.dump(audit, f, indent=2, ensure_ascii=False)
    with open(out_dir / "scientific_verdict.json", "w", encoding="utf-8") as f:
        json.dump({"verdict": verdict, "rationale": rationale}, f, indent=2, ensure_ascii=False)
    with open(out_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    return {
        "metrics_df": metrics_df,
        "summary_df": summary,
        "reliability_df": reliability_df,
        "perturbation_df": perturb_df,
        "audit": audit,
        "manifest": manifest,
    }


if __name__ == "__main__":
    cfg = BenchmarkConfig()
    out = run_benchmark(cfg)
    print("Done. Verdict:", out["audit"]["verdict"])
'''
target.write_text(textwrap.dedent(code), encoding="utf-8")
print(f"[OK] wrote: {target}")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/cognitive-self-model")
p = repo / "reports/M21_2_4_3_4/input_dataset.csv"
p.parent.mkdir(parents=True, exist_ok=True)

if not p.exists():
    rng = np.random.default_rng(42)
    n = 1200
    episodes = [f"ep_{i:03d}" for i in range(60)]
    tracks = ["IN_DISTRIBUTION", "REGIME_CHANGE_IN_SCOPE", "NOVEL_BUT_VALID", "OOD_INVALID"]

    df = pd.DataFrame({
        "episode_id": rng.choice(episodes, size=n, replace=True),
        "track": rng.choice(tracks, size=n, replace=True, p=[0.45, 0.2, 0.2, 0.15]),
    })

    z = rng.normal(0, 1, size=n)
    z += (df["track"].eq("OOD_INVALID").astype(float) * 1.1)
    z -= (df["track"].eq("NOVEL_BUT_VALID").astype(float) * 0.5)

    q = 1.0 / (1.0 + np.exp(-z))
    df["q_invalid"] = np.clip(q, 1e-6, 1-1e-6)

    py = np.clip(0.08 + 0.75 * df["q_invalid"], 0, 1)
    df["self_model_invalid"] = rng.binomial(1, py).astype(int)

    df.to_csv(p, index=False)
    print(f"[OK] created: {p} | shape={df.shape}")
else:
    print(f"[OK] exists: {p}")

In [ ]:
import sys
from pathlib import Path

repo = Path("/content/cognitive-self-model")
sys.path.append(str(repo / "src"))

from legacy_import.m21_2_4_3_4_identifiability_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    input_csv=str(repo / "reports/M21_2_4_3_4/input_dataset.csv"),
    output_dir=str(repo / "reports/M21_2_4_3_4"),
    n_folds=5,
    bootstrap_repeats=120,
    reliability_bins=10,
    perturbation_eps=0.02,
)
out = run_benchmark(cfg)
print("Verdict:", out["audit"]["verdict"])
print(out["summary_df"])

In [ ]:
import json
from pathlib import Path
import pandas as pd

repo = Path("/content/cognitive-self-model")
od = repo / "reports/M21_2_4_3_4"

print("Files:")
for p in sorted(od.glob("*")):
    print("-", p.name)

print("\nscientific_verdict.json:")
print(json.loads((od / "scientific_verdict.json").read_text(encoding="utf-8")))

print("\nreliability sample:")
print(pd.read_csv(od / "reliability_by_method.csv").head())

print("\nperturbation stress:")
print(pd.read_csv(od / "perturbation_stress.csv"))

print("\nnegative control:")
print(json.loads((od / "negative_control_results.json").read_text(encoding="utf-8")))

print("\nsample identity ablation:")
print(json.loads((od / "sample_identity_ablation.json").read_text(encoding="utf-8")))

In [ ]:
import json
from pathlib import Path

p = Path("/content/cognitive-self-model/reports/M21_2_4_3_4/scientific_verdict.json")
print(json.loads(p.read_text(encoding="utf-8")))

In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import re
from collections import Counter

REPO_ROOT = Path("/content/cognitive-self-model").resolve()

SEARCH_ROOTS = [
    REPO_ROOT / "artifacts",
    REPO_ROOT / "reports",
    REPO_ROOT / "data",
    REPO_ROOT / "configs",
    REPO_ROOT / "src",
    REPO_ROOT / "notebooks",
]

KEYWORDS = (
    "m21",
    "2_4_2",
    "invalid",
    "q_invalid",
    "episode",
    "self_model",
    "track",
    "estimator",
    "calibration",
)

TEXT_SUFFIXES = {".py", ".ipynb", ".md", ".txt", ".json", ".jsonl", ".yaml", ".yml", ".csv"}
DATA_SUFFIXES = {".csv", ".json", ".jsonl", ".parquet"}

MAX_TEXT_BYTES = 2_000_000
MAX_JSONL_RECORDS = 3
MAX_RESULTS = 500

def safe_rel(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)

def read_csv_header(path: Path) -> list[str] | None:
    try:
        with path.open("r", encoding="utf-8-sig", newline="") as f:
            return next(csv.reader(f), None)
    except Exception as exc:
        return [f"<csv-read-error: {type(exc).__name__}: {exc}>"]

def read_json_summary(path: Path) -> dict:
    try:
        if path.stat().st_size > MAX_TEXT_BYTES:
            return {"status": "skipped_large_file", "size_bytes": path.stat().st_size}

        obj = json.loads(path.read_text(encoding="utf-8"))

        if isinstance(obj, dict):
            return {
                "type": "dict",
                "top_level_keys": sorted(map(str, obj.keys()))[:100],
            }
        if isinstance(obj, list):
            first = obj[0] if obj else None
            return {
                "type": "list",
                "length": len(obj),
                "first_item_keys": (
                    sorted(map(str, first.keys()))[:100]
                    if isinstance(first, dict)
                    else None
                ),
            }
        return {"type": type(obj).__name__}
    except Exception as exc:
        return {"status": f"json-read-error: {type(exc).__name__}: {exc}"}

def read_jsonl_summary(path: Path) -> dict:
    records = []
    try:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                records.append(json.loads(line))
                if len(records) >= MAX_JSONL_RECORDS:
                    break

        keys = Counter()
        for record in records:
            if isinstance(record, dict):
                keys.update(map(str, record.keys()))

        return {
            "sampled_records": len(records),
            "sampled_keys": sorted(keys.keys())[:100],
        }
    except Exception as exc:
        return {"status": f"jsonl-read-error: {type(exc).__name__}: {exc}"}

def keyword_hits(path: Path) -> list[str]:
    if path.suffix.lower() not in TEXT_SUFFIXES:
        return []

    try:
        if path.stat().st_size > MAX_TEXT_BYTES:
            return []

        text = path.read_text(encoding="utf-8", errors="replace").lower()
        return [kw for kw in KEYWORDS if kw in text]
    except Exception:
        return []

candidates = []
all_files = []

for root in SEARCH_ROOTS:
    if not root.exists():
        print(f"[MISSING] {safe_rel(root)}")
        continue

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        all_files.append(path)
        suffix = path.suffix.lower()
        name_lower = path.name.lower()
        hits = keyword_hits(path)

        name_match = any(kw in name_lower for kw in KEYWORDS)
        if name_match or hits or suffix in DATA_SUFFIXES:
            item = {
                "path": safe_rel(path),
                "size_bytes": path.stat().st_size,
                "suffix": suffix,
                "name_keyword_match": name_match,
                "content_keyword_hits": hits,
            }

            if suffix == ".csv":
                item["schema"] = read_csv_header(path)
            elif suffix == ".json":
                item["schema"] = read_json_summary(path)
            elif suffix == ".jsonl":
                item["schema"] = read_jsonl_summary(path)

            candidates.append(item)

print(f"\nRepository root: {REPO_ROOT}")
print(f"Files scanned: {len(all_files)}")
print(f"Candidate/data files found: {len(candidates)}\n")

for i, item in enumerate(candidates[:MAX_RESULTS], start=1):
    print(f"{i:03d}. {item['path']}")
    print(f"     size_bytes={item['size_bytes']} | suffix={item['suffix']}")
    if item["content_keyword_hits"]:
        print(f"     keyword_hits={item['content_keyword_hits']}")
    if "schema" in item:
        print(f"     schema={item['schema']}")

if len(candidates) > MAX_RESULTS:
    print(f"\n[TRUNCATED] Showing first {MAX_RESULTS} of {len(candidates)} candidates.")

In [ ]:
%cd /content/cognitive-self-model
!git status --short
!git log -5 --oneline

In [ ]:
from __future__ import annotations

from pathlib import Path
import getpass
import os
import shutil
import subprocess
import textwrap

REPOSITORY_URL = "https://github.com/motiza345/cognitive-self-model.git"
BRANCH_NAME = "chore/repository-scientific-structure"
CLONE_DIR = Path("/content/cognitive-self-model-scientific-structure").resolve()
ASKPASS_PATH = Path("/content/git_askpass_cognitive_self_model.sh").resolve()

if CLONE_DIR.exists():
    print(f"[STOP] Clone target already exists: {CLONE_DIR}")
    print("No file was changed. Inspect this directory before deciding whether it can be reused.")
else:
    github_username = input("GitHub username: ").strip()
    github_token = getpass.getpass("GitHub Fine-grained PAT (repository read access): ").strip()

    if not github_username or not github_token:
        raise ValueError("GitHub username and token are both required for private repository access.")

    ASKPASS_PATH.write_text(
        textwrap.dedent(
            """\
            #!/bin/sh
            case "$1" in
              *Username*) printf '%s\\n' "$GITHUB_USERNAME" ;;
              *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
              *) printf '\\n' ;;
            esac
            """
        ),
        encoding="utf-8",
    )
    ASKPASS_PATH.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(ASKPASS_PATH)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GITHUB_USERNAME"] = github_username
    env["GITHUB_TOKEN"] = github_token

    try:
        result = subprocess.run(
            [
                "git",
                "clone",
                "--branch", BRANCH_NAME,
                "--single-branch",
                "--depth", "1",
                REPOSITORY_URL,
                str(CLONE_DIR),
            ],
            text=True,
            capture_output=True,
            env=env,
            check=False,
        )

        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f"git clone failed with return code {result.returncode}.")

        print(f"[OK] Repository cloned to: {CLONE_DIR}")

    finally:
        os.environ.pop("GITHUB_TOKEN", None)
        os.environ.pop("GITHUB_USERNAME", None)
        if ASKPASS_PATH.exists():
            ASKPASS_PATH.unlink()

In [ ]:
REPO_ROOT = Path("/content/cognitive-self-model-scientific-structure").resolve()

In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import Counter
import csv
import json
import subprocess

# -----------------------------------------------------------------------------
# Repository target
# -----------------------------------------------------------------------------

REPO_ROOT = Path("/content/cognitive-self-model-scientific-structure").resolve()

IMPORTANT_DIRS = [
    "artifacts",
    "configs",
    "data",
    "notebooks",
    "reports",
    "src",
    "tests",
]

TEXT_SUFFIXES = {
    ".py", ".ipynb", ".json", ".yaml", ".yml", ".toml",
    ".md", ".txt", ".csv", ".parquet",
}

SEARCH_TERMS = [
    "M21.2.4.2",
    "M21_2_4_2",
    "M21.2.4.3.3",
    "M21_2_4_3_3",
    "M21.2.4.3.4",
    "M21_2_4_3_4",
    "q_invalid",
    "self_model_invalid",
    "episode_id",
    "track",
    "calibration",
    "isotonic",
    "platt",
    "identifiability",
]

MAX_FILE_SIZE_FOR_TEXT_SCAN = 2_000_000
MAX_CSV_SCHEMA_COLUMNS = 100
MAX_RESULTS_PER_TERM = 80


# -----------------------------------------------------------------------------
# Read-only helpers
# -----------------------------------------------------------------------------

def run_git(*args: str) -> tuple[int, str, str]:
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def relative_path(path: Path) -> str:
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def safe_iterdir(path: Path) -> list[Path]:
    try:
        return sorted(path.iterdir(), key=lambda item: item.name.lower())
    except (FileNotFoundError, PermissionError, OSError):
        return []


def safe_read_text(path: Path, max_bytes: int = MAX_FILE_SIZE_FOR_TEXT_SCAN) -> str | None:
    try:
        if path.stat().st_size > max_bytes:
            return None
        return path.read_text(encoding="utf-8", errors="replace")
    except (FileNotFoundError, PermissionError, OSError, UnicodeDecodeError):
        return None


def csv_schema(path: Path) -> list[str] | None:
    try:
        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle)
            header = next(reader, None)
        if not header:
            return []
        return [column.strip() for column in header[:MAX_CSV_SCHEMA_COLUMNS]]
    except (FileNotFoundError, PermissionError, OSError, UnicodeDecodeError, csv.Error):
        return None


def json_top_level_schema(path: Path) -> str | None:
    try:
        if path.stat().st_size > MAX_FILE_SIZE_FOR_TEXT_SCAN:
            return "<skipped: file too large>"
        with path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)

        if isinstance(payload, dict):
            keys = list(payload.keys())
            return f"dict keys={keys[:30]}"
        if isinstance(payload, list):
            return f"list length={len(payload)}"
        return f"type={type(payload).__name__}"
    except (FileNotFoundError, PermissionError, OSError, UnicodeDecodeError, json.JSONDecodeError):
        return None


def contains_candidate_columns(columns: list[str]) -> bool:
    normalized = {column.strip().lower() for column in columns}
    required_signal_columns = {
        "episode_id",
        "q_invalid",
        "self_model_invalid",
        "track",
    }
    return bool(normalized & required_signal_columns)


# -----------------------------------------------------------------------------
# Preconditions and Git identity
# -----------------------------------------------------------------------------

print("=" * 100)
print("READ-ONLY REPOSITORY DISCOVERY")
print("=" * 100)
print(f"Repository root: {REPO_ROOT}")
print(f"Exists: {REPO_ROOT.exists()}")
print(f"Git metadata exists: {(REPO_ROOT / '.git').exists()}")

if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Repository directory does not exist: {REPO_ROOT}\n"
        "Update REPO_ROOT to the actual cloned repository path."
    )

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        f"{REPO_ROOT} is not a Git repository. "
        "Do not treat this directory as the authoritative project clone."
    )

print("\n--- Git identity ---")
for label, args in [
    ("origin", ("remote", "get-url", "origin")),
    ("branch", ("branch", "--show-current")),
    ("head", ("log", "-1", "--format=%H%n%ad%n%s", "--date=iso-strict")),
    ("status", ("status", "--short")),
]:
    code, stdout, stderr = run_git(*args)
    value = stdout if code == 0 else f"<unavailable: {stderr}>"
    if label == "status" and code == 0 and not value:
        value = "<clean>"
    print(f"{label}: {value}")

# -----------------------------------------------------------------------------
# Top-level and required-directory inventory
# -----------------------------------------------------------------------------

print("\n" + "-" * 100)
print("TOP-LEVEL INVENTORY")
print("-" * 100)

for item in safe_iterdir(REPO_ROOT):
    if item.name == ".git":
        continue
    kind = "DIR " if item.is_dir() else "FILE"
    print(f"{kind} {item.name}")

print("\n" + "-" * 100)
print("EXPECTED SCIENTIFIC DIRECTORIES")
print("-" * 100)

for directory_name in IMPORTANT_DIRS:
    directory = REPO_ROOT / directory_name
    if directory.is_dir():
        children = safe_iterdir(directory)
        print(f"[FOUND]   {directory_name}/  ({len(children)} immediate entries)")
    else:
        print(f"[MISSING] {directory_name}/")

# -----------------------------------------------------------------------------
# File inventory
# -----------------------------------------------------------------------------

all_files: list[Path] = []
for path in REPO_ROOT.rglob("*"):
    if not path.is_file():
        continue
    if ".git" in path.parts or "__pycache__" in path.parts:
        continue
    all_files.append(path)

suffix_counts = Counter(
    path.suffix.lower() if path.suffix else "<no_suffix>"
    for path in all_files
)

print("\n" + "-" * 100)
print("FILE INVENTORY")
print("-" * 100)
print(f"Files scanned: {len(all_files)}")
print("Suffix counts:")
for suffix, count in suffix_counts.most_common():
    print(f"  {suffix}: {count}")

# -----------------------------------------------------------------------------
# Milestone and semantic-term discovery
# -----------------------------------------------------------------------------

print("\n" + "-" * 100)
print("MILESTONE / SEMANTIC REFERENCE DISCOVERY")
print("-" * 100)

matches_by_term: dict[str, list[str]] = {term: [] for term in SEARCH_TERMS}

for path in all_files:
    relative = relative_path(path)
    path_name_lower = relative.lower()

    # Match filename/path first.
    for term in SEARCH_TERMS:
        if term.lower() in path_name_lower:
            if relative not in matches_by_term[term]:
                matches_by_term[term].append(relative)

    # Then inspect text-like, reasonably sized files.
    if path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    content = safe_read_text(path)
    if content is None:
        continue

    content_lower = content.lower()
    for term in SEARCH_TERMS:
        if term.lower() in content_lower and relative not in matches_by_term[term]:
            matches_by_term[term].append(relative)

for term in SEARCH_TERMS:
    matches = sorted(matches_by_term[term])
    print(f"\n[{term}] matches={len(matches)}")
    if not matches:
        print("  <none>")
        continue
    for match in matches[:MAX_RESULTS_PER_TERM]:
        print(f"  - {match}")
    if len(matches) > MAX_RESULTS_PER_TERM:
        print(f"  ... {len(matches) - MAX_RESULTS_PER_TERM} additional matches omitted")

# -----------------------------------------------------------------------------
# Candidate dataset / artifact inspection
# -----------------------------------------------------------------------------

print("\n" + "-" * 100)
print("CANDIDATE DATASET AND ARTIFACT INSPECTION")
print("-" * 100)

candidate_files: list[Path] = []

for path in all_files:
    relative_lower = relative_path(path).lower()
    suffix = path.suffix.lower()

    looks_data_like = suffix in {".csv", ".json", ".parquet", ".xlsx", ".xls"}
    has_relevant_name = any(
        token in relative_lower
        for token in [
            "m21",
            "invalid",
            "calibration",
            "episode",
            "dataset",
            "artifact",
            "metric",
            "manifest",
            "audit",
        ]
    )

    if looks_data_like and has_relevant_name:
        candidate_files.append(path)

# Ensure CSV files with target schema are included even if their names are generic.
for path in all_files:
    if path.suffix.lower() != ".csv" or path in candidate_files:
        continue
    schema = csv_schema(path)
    if schema and contains_candidate_columns(schema):
        candidate_files.append(path)

candidate_files = sorted(set(candidate_files), key=lambda item: relative_path(item).lower())

if not candidate_files:
    print("<No candidate data/artifact files found>")
else:
    for index, path in enumerate(candidate_files, start=1):
        relative = relative_path(path)
        try:
            size_bytes = path.stat().st_size
        except OSError:
            size_bytes = -1

        print(f"\n{index:03d}. {relative}")
        print(f"     size_bytes={size_bytes} | suffix={path.suffix.lower() or '<none>'}")

        if path.suffix.lower() == ".csv":
            schema = csv_schema(path)
            print(f"     csv_schema={schema if schema is not None else '<unreadable>'}")

        elif path.suffix.lower() == ".json":
            schema = json_top_level_schema(path)
            print(f"     json_schema={schema if schema is not None else '<unreadable>'}")

# -----------------------------------------------------------------------------
# Direct search for source files that define or produce required fields
# -----------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PROVENANCE LEADS FOR REQUIRED BENCHMARK FIELDS")
print("-" * 100)

required_fields = [
    "episode_id",
    "q_invalid",
    "self_model_invalid",
    "track",
]

source_suffixes = {".py", ".ipynb", ".json", ".yaml", ".yml", ".toml", ".md"}
field_hits: dict[str, list[str]] = {field: [] for field in required_fields}

for path in all_files:
    if path.suffix.lower() not in source_suffixes:
        continue

    content = safe_read_text(path)
    if content is None:
        continue

    content_lower = content.lower()
    relative = relative_path(path)

    for field in required_fields:
        if field.lower() in content_lower:
            field_hits[field].append(relative)

for field in required_fields:
    hits = sorted(set(field_hits[field]))
    print(f"\nField: {field} | files={len(hits)}")
    if not hits:
        print("  <no provenance lead found>")
        continue
    for hit in hits[:MAX_RESULTS_PER_TERM]:
        print(f"  - {hit}")
    if len(hits) > MAX_RESULTS_PER_TERM:
        print(f"  ... {len(hits) - MAX_RESULTS_PER_TERM} additional files omitted")

print("\n" + "=" * 100)
print("DISCOVERY COMPLETE")
print("=" * 100)
print(
    "Next audit target: identify the authoritative source and generation path for "
    "episode_id, q_invalid, self_model_invalid, and track before treating any CSV "
    "as a scientifically valid M21.2.4.3.4 input dataset."
)

In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import subprocess
from typing import Any

# =============================================================================
# READ-ONLY M21.2.4.3.4 PROVENANCE AND CONTRACT AUDIT
# =============================================================================

REPO_ROOT = Path("/content/cognitive-self-model-scientific-structure").resolve()

CONFIG_PATH = REPO_ROOT / "configs" / "m21_2_4_3_4_config.json"
BENCHMARK_SOURCE_PATH = (
    REPO_ROOT
    / "src"
    / "legacy_import"
    / "m21_2_4_3_4_identifiability_benchmark.py"
)
AUDIT_REPORT_PATH = (
    REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "m21_2_4_3_4_identifiability_audit.json"
)
VERDICT_PATH = (
    REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "scientific_verdict.json"
)
FOLD_METRICS_PATH = (
    REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "fold_metrics.csv"
)
GITIGNORE_PATH = REPO_ROOT / ".gitignore"

# Candidate local paths only. No recursive raw-data dump is performed.
INPUT_CANDIDATES = [
    REPO_ROOT / "reports" / "M21_2_4_3_4" / "input_dataset.csv",
    REPO_ROOT / "artifacts" / "M21_2_4_3_4" / "input_dataset.csv",
    REPO_ROOT / "artifacts" / "M21_2_4_3_4" / "benchmark_input.csv",
    REPO_ROOT / "data" / "M21_2_4_3_4" / "input_dataset.csv",
]

MAX_SOURCE_LINES = 800
MAX_JSON_CHARS = 20_000


def run_git(*args: str) -> tuple[int, str, str]:
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def print_heading(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def print_json_file(path: Path) -> dict[str, Any] | None:
    print(f"Path: {path.relative_to(REPO_ROOT)}")
    print(f"Exists: {path.exists()}")

    if not path.exists():
        return None

    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[UNREADABLE JSON] {type(exc).__name__}: {exc}")
        return None

    rendered = json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=False)
    if len(rendered) > MAX_JSON_CHARS:
        rendered = rendered[:MAX_JSON_CHARS] + "\n... <truncated>"
    print(rendered)
    return payload


def csv_header_and_row_count(path: Path) -> tuple[list[str] | None, int | None]:
    try:
        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle)
            header = next(reader, None)
            if header is None:
                return [], 0
            row_count = sum(1 for _ in reader)
        return [value.strip() for value in header], row_count
    except Exception:
        return None, None


def git_file_state(path: Path) -> str:
    """
    Reports tracked / ignored / untracked status without changing the repository.
    """
    relative = str(path.relative_to(REPO_ROOT))

    tracked_code, tracked_out, _ = run_git("ls-files", "--error-unmatch", "--", relative)
    if tracked_code == 0:
        return "TRACKED"

    ignored_code, ignored_out, _ = run_git("check-ignore", "-v", "--", relative)
    if ignored_code == 0:
        return f"IGNORED ({ignored_out})"

    if path.exists():
        return "UNTRACKED_OR_NOT_IN_INDEX"

    return "ABSENT"


# -----------------------------------------------------------------------------
# 1. Git identity
# -----------------------------------------------------------------------------

print_heading("1. GIT IDENTITY")

for label, args in [
    ("origin", ("remote", "get-url", "origin")),
    ("branch", ("branch", "--show-current")),
    ("head", ("log", "-1", "--format=%H%n%ad%n%s", "--date=iso-strict")),
    ("status", ("status", "--short")),
]:
    code, stdout, stderr = run_git(*args)
    value = stdout if code == 0 else f"<unavailable: {stderr}>"
    if label == "status" and code == 0 and not value:
        value = "<clean>"
    print(f"{label}: {value}")

# -----------------------------------------------------------------------------
# 2. Config: authoritative declared design
# -----------------------------------------------------------------------------

print_heading("2. M21.2.4.3.4 CONFIGURATION")
config_payload = print_json_file(CONFIG_PATH)

# -----------------------------------------------------------------------------
# 3. Existing audit report and verdict
# -----------------------------------------------------------------------------

print_heading("3. EXISTING IDENTIFIABILITY AUDIT REPORT")
audit_payload = print_json_file(AUDIT_REPORT_PATH)

print_heading("4. EXISTING SCIENTIFIC VERDICT")
verdict_payload = print_json_file(VERDICT_PATH)

# -----------------------------------------------------------------------------
# 4. Existing fold metrics
# -----------------------------------------------------------------------------

print_heading("5. EXISTING FOLD METRICS")
print(f"Path: {FOLD_METRICS_PATH.relative_to(REPO_ROOT)}")
print(f"Exists: {FOLD_METRICS_PATH.exists()}")

if FOLD_METRICS_PATH.exists():
    header, row_count = csv_header_and_row_count(FOLD_METRICS_PATH)
    print(f"Header: {header}")
    print(f"Data rows: {row_count}")
    print("\nContents:")
    print(FOLD_METRICS_PATH.read_text(encoding="utf-8-sig", errors="replace"))
else:
    print("<missing>")

# -----------------------------------------------------------------------------
# 5. Input path declared by report, plus candidate-path state
# -----------------------------------------------------------------------------

print_heading("6. INPUT DATASET PRESENCE AND GIT STATE")

declared_input_path: Path | None = None
if isinstance(audit_payload, dict) and audit_payload.get("input_csv"):
    raw_input = Path(str(audit_payload["input_csv"]))
    declared_input_path = raw_input if raw_input.is_absolute() else REPO_ROOT / raw_input
    INPUT_CANDIDATES.insert(0, declared_input_path)

unique_candidates: list[Path] = []
seen: set[Path] = set()

for candidate in INPUT_CANDIDATES:
    resolved = candidate.resolve()
    if resolved not in seen:
        unique_candidates.append(resolved)
        seen.add(resolved)

for path in unique_candidates:
    try:
        relative = path.relative_to(REPO_ROOT)
        display_path = str(relative)
        state = git_file_state(path)
    except ValueError:
        display_path = str(path)
        state = "OUTSIDE_REPOSITORY"

    print(f"\nCandidate: {display_path}")
    print(f"Exists: {path.exists()}")
    print(f"Git state: {state}")

    if path.exists() and path.suffix.lower() == ".csv":
        header, row_count = csv_header_and_row_count(path)
        print(f"CSV header: {header}")
        print(f"CSV data rows: {row_count}")

        required_columns = {"episode_id", "q_invalid", "self_model_invalid", "track"}
        observed_columns = set(header or [])
        print(f"Required columns present: {required_columns.issubset(observed_columns)}")
        print(f"Missing required columns: {sorted(required_columns - observed_columns)}")

# -----------------------------------------------------------------------------
# 6. Ignore policy, especially the ignored benchmark input
# -----------------------------------------------------------------------------

print_heading("7. .GITIGNORE POLICY")
print(f"Path: {GITIGNORE_PATH.relative_to(REPO_ROOT)}")
print(f"Exists: {GITIGNORE_PATH.exists()}")

if GITIGNORE_PATH.exists():
    print(GITIGNORE_PATH.read_text(encoding="utf-8", errors="replace"))
else:
    print("<missing>")

# -----------------------------------------------------------------------------
# 7. Source-code contract extraction
# -----------------------------------------------------------------------------

print_heading("8. BENCHMARK SOURCE: CONTRACT AND LABEL-PROVENANCE LEADS")
print(f"Path: {BENCHMARK_SOURCE_PATH.relative_to(REPO_ROOT)}")
print(f"Exists: {BENCHMARK_SOURCE_PATH.exists()}")

if BENCHMARK_SOURCE_PATH.exists():
    source_lines = BENCHMARK_SOURCE_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    search_terms = [
        "read_csv",
        "input_csv",
        "episode_id",
        "q_invalid",
        "self_model_invalid",
        "invalid",
        "label",
        "target",
        "track",
        "forbidden",
        "groundtruth",
        "verdict",
        "SUPPORTED_CALIBRATION",
        "CONDITIONAL_SUPPORT_LIMITED",
        "CALIBRATION_NOT_STABLE_OR_NOT_IDENTIFIABLE",
    ]

    matching_line_numbers: set[int] = set()
    for index, line in enumerate(source_lines, start=1):
        line_lower = line.lower()
        if any(term.lower() in line_lower for term in search_terms):
            matching_line_numbers.add(index)

    context_line_numbers: set[int] = set()
    for line_number in matching_line_numbers:
        start = max(1, line_number - 2)
        end = min(len(source_lines), line_number + 2)
        context_line_numbers.update(range(start, end + 1))

    print(f"Total source lines: {len(source_lines)}")
    print(f"Matched source lines: {len(matching_line_numbers)}")
    print("\n--- Relevant source contexts ---")

    for line_number in sorted(context_line_numbers):
        marker = ">>" if line_number in matching_line_numbers else "  "
        print(f"{marker} {line_number:04d}: {source_lines[line_number - 1]}")

    if len(source_lines) > MAX_SOURCE_LINES:
        print(f"\n<Source has more than {MAX_SOURCE_LINES} lines; only relevant contexts shown.>")
else:
    print("<missing>")

# -----------------------------------------------------------------------------
# 8. Repository index check for all M21.2.4.3.4 files
# -----------------------------------------------------------------------------

print_heading("9. GIT-TRACKED M21.2.4.3.4 FILES")
code, stdout, stderr = run_git("ls-files", "*M21_2_4_3_4*", "*m21_2_4_3_4*")

if code == 0 and stdout:
    print(stdout)
elif code == 0:
    print("<no matching tracked paths>")
else:
    print(f"<git ls-files failed: {stderr}>")

# -----------------------------------------------------------------------------
# 9. Conclusion guard
# -----------------------------------------------------------------------------

print_heading("10. AUDIT GUARD")

print(
    "A final scientific verdict is provenance-supported only if the benchmark input "
    "exists, satisfies the declared CSV contract, and has a documented reproducible "
    "generation path from authoritative project artifacts. An ignored local CSV may "
    "be privacy-preserving, but it is not independently auditable from Git alone."
)

In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import math
import subprocess

import pandas as pd


# =============================================================================
# READ-ONLY M21.2.4.3.4 EXTERNAL-INPUT PROVENANCE + CONTRACT RECONCILIATION
# =============================================================================

REPO_ROOT = Path("/content/cognitive-self-model-scientific-structure").resolve()

AUDIT_REPORT_PATH = (
    REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "m21_2_4_3_4_identifiability_audit.json"
)
RUNNER_PATH = (
    REPO_ROOT
    / "src"
    / "legacy_import"
    / "m21_2_4_3_4_identifiability_benchmark.py"
)
FOLD_METRICS_PATH = (
    REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "fold_metrics.csv"
)

RUNNER_REQUIRED_COLUMNS = [
    "episode_id",
    "split_role",
    "raw_q_invalid",
    "label_invalid",
    "track",
]

CURRENT_EXTERNAL_SCHEMA = [
    "episode_id",
    "track",
    "q_invalid",
    "self_model_invalid",
]


def heading(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def run_git(*args: str) -> tuple[int, str, str]:
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def safe_numeric_summary(series: pd.Series) -> dict[str, float | int | None]:
    numeric = pd.to_numeric(series, errors="coerce")
    finite = numeric[numeric.map(lambda value: math.isfinite(value) if pd.notna(value) else False)]

    if finite.empty:
        return {
            "n_total": int(len(series)),
            "n_missing_or_nonfinite": int(len(series)),
            "min": None,
            "max": None,
            "mean": None,
            "n_below_0": None,
            "n_above_1": None,
        }

    return {
        "n_total": int(len(series)),
        "n_missing_or_nonfinite": int(len(series) - len(finite)),
        "min": float(finite.min()),
        "max": float(finite.max()),
        "mean": float(finite.mean()),
        "n_below_0": int((finite < 0).sum()),
        "n_above_1": int((finite > 1).sum()),
    }


# -----------------------------------------------------------------------------
# 1. Resolve authoritative path declared by the existing audit manifest
# -----------------------------------------------------------------------------

heading("1. DECLARED EXTERNAL INPUT PATH")

if not AUDIT_REPORT_PATH.exists():
    raise FileNotFoundError(f"Missing audit report: {AUDIT_REPORT_PATH}")

audit_payload = json.loads(AUDIT_REPORT_PATH.read_text(encoding="utf-8"))
declared_input = audit_payload.get("input_csv")

if not declared_input:
    raise ValueError("The audit report has no 'input_csv' field.")

EXTERNAL_INPUT_PATH = Path(declared_input)

print(f"Manifest path: {AUDIT_REPORT_PATH.relative_to(REPO_ROOT)}")
print(f"Declared input_csv: {EXTERNAL_INPUT_PATH}")
print(f"Exists: {EXTERNAL_INPUT_PATH.exists()}")
print(f"Inside current repository: {REPO_ROOT in EXTERNAL_INPUT_PATH.resolve().parents}")

if not EXTERNAL_INPUT_PATH.exists():
    print(
        "\nSTOP: the exact external input declared by the existing manifest is absent. "
        "Its provenance and contents cannot be audited from this runtime."
    )
else:
    # -------------------------------------------------------------------------
    # 2. Privacy-safe input inventory
    # -------------------------------------------------------------------------

    heading("2. PRIVACY-SAFE EXTERNAL INPUT INVENTORY")

    print(f"File size bytes: {EXTERNAL_INPUT_PATH.stat().st_size}")
    print(f"SHA256: {sha256_file(EXTERNAL_INPUT_PATH)}")

    with EXTERNAL_INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader, None)

    print(f"CSV header: {header}")

    df = pd.read_csv(EXTERNAL_INPUT_PATH)

    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Duplicate full rows: {int(df.duplicated().sum())}")
    print(f"Missing values by column: {df.isna().sum().astype(int).to_dict()}")

    # No raw identifiers, rows, or values are printed.
    if "episode_id" in df.columns:
        episode_count = int(df["episode_id"].nunique(dropna=True))
        rows_per_episode = df.groupby("episode_id", dropna=False).size()

        print(f"Unique episode_id count: {episode_count}")
        print(f"Rows per episode — min: {int(rows_per_episode.min())}")
        print(f"Rows per episode — median: {float(rows_per_episode.median())}")
        print(f"Rows per episode — max: {int(rows_per_episode.max())}")
        print(f"Episodes with more than one row: {int((rows_per_episode > 1).sum())}")

    if "track" in df.columns:
        track_counts = df["track"].value_counts(dropna=False)
        print(f"Unique track count: {int(track_counts.shape[0])}")
        print(f"Track row counts: {track_counts.to_dict()}")

        if "episode_id" in df.columns:
            episode_track_counts = (
                df.groupby("track", dropna=False)["episode_id"]
                .nunique(dropna=True)
                .astype(int)
                .to_dict()
            )
            print(f"Track unique-episode counts: {episode_track_counts}")

    for probability_column in ["q_invalid", "raw_q_invalid"]:
        if probability_column in df.columns:
            print(
                f"{probability_column} summary: "
                f"{safe_numeric_summary(df[probability_column])}"
            )

    for label_column in ["self_model_invalid", "label_invalid"]:
        if label_column in df.columns:
            normalized = pd.to_numeric(df[label_column], errors="coerce")
            label_counts = Counter(normalized.dropna().astype(str).tolist())

            print(f"{label_column} value counts: {dict(sorted(label_counts.items()))}")
            print(
                f"{label_column} is binary 0/1: "
                f"{set(normalized.dropna().unique()).issubset({0, 1})}"
            )

    # -------------------------------------------------------------------------
    # 3. Contract reconciliation
    # -------------------------------------------------------------------------

    heading("3. RUNNER CONTRACT RECONCILIATION")

    observed_columns = set(df.columns)
    runner_required = set(RUNNER_REQUIRED_COLUMNS)

    print(f"Runner required columns: {RUNNER_REQUIRED_COLUMNS}")
    print(f"Observed columns: {list(df.columns)}")
    print(f"Missing for current runner: {sorted(runner_required - observed_columns)}")
    print(f"Extra versus current runner: {sorted(observed_columns - runner_required)}")
    print(f"Current runner can directly consume input: {runner_required.issubset(observed_columns)}")

    semantic_aliases = {
        "raw_q_invalid": "q_invalid",
        "label_invalid": "self_model_invalid",
    }

    print("\nPotential semantic aliases detected (NOT applied):")
    for runner_column, input_column in semantic_aliases.items():
        print(
            f"  {runner_column} <- {input_column}: "
            f"{input_column in observed_columns}"
        )

    print(
        "\nIMPORTANT: alias availability does not establish semantic equivalence. "
        "No rename, conversion, split creation, or mutation was performed."
    )

    # -------------------------------------------------------------------------
    # 4. Reconcile old metric row against source input
    # -------------------------------------------------------------------------

    heading("4. FOLD-METRICS RECONCILIATION")

    if not FOLD_METRICS_PATH.exists():
        print(f"Missing metrics file: {FOLD_METRICS_PATH}")
    else:
        metrics_df = pd.read_csv(FOLD_METRICS_PATH)
        print(f"Metrics rows: {len(metrics_df)}")
        print(f"Metrics columns: {list(metrics_df.columns)}")

        if "n" in metrics_df.columns and len(metrics_df) > 0:
            metric_n_values = pd.to_numeric(metrics_df["n"], errors="coerce").dropna()
            metric_n = int(metric_n_values.iloc[0]) if not metric_n_values.empty else None

            print(f"Recorded metric n (first row): {metric_n}")
            print(f"External input row count: {len(df)}")
            print(f"Metric n equals external input rows: {metric_n == len(df)}")

            if metric_n != len(df):
                print(
                    "RECONCILIATION FAILURE: recorded fold metric n cannot have been "
                    "computed directly from this exact external CSV as one-row-per-input-row "
                    "baseline output."
                )

    # -------------------------------------------------------------------------
    # 5. Git history relevant to provenance
    # -------------------------------------------------------------------------

    heading("5. GIT HISTORY FOR CURRENT M21.2.4.3.4 TRACKED FILES")

    tracked_paths = [
        "configs/m21_2_4_3_4_config.json",
        "src/legacy_import/m21_2_4_3_4_identifiability_benchmark.py",
        "reports/M21_2_4_3_4/fold_metrics.csv",
        "reports/M21_2_4_3_4/m21_2_4_3_4_identifiability_audit.json",
        "reports/M21_2_4_3_4/scientific_verdict.json",
    ]

    for relative_path in tracked_paths:
        print(f"\n--- {relative_path} ---")
        code, stdout, stderr = run_git(
            "log",
            "--follow",
            "--format=%H | %ad | %s",
            "--date=iso-strict",
            "-n",
            "10",
            "--",
            relative_path,
        )
        print(stdout if code == 0 and stdout else f"<no history or error: {stderr}>")

    # -------------------------------------------------------------------------
    # 6. Final machine-readable status
    # -------------------------------------------------------------------------

    heading("6. AUDIT STATUS")

    direct_runner_compatible = set(RUNNER_REQUIRED_COLUMNS).issubset(set(df.columns))

    metric_n_matches = False
    if FOLD_METRICS_PATH.exists():
        metrics_df = pd.read_csv(FOLD_METRICS_PATH)
        if "n" in metrics_df.columns and len(metrics_df) > 0:
            values = pd.to_numeric(metrics_df["n"], errors="coerce").dropna()
            metric_n_matches = bool(not values.empty and int(values.iloc[0]) == len(df))

    status = {
        "external_input_exists": True,
        "external_input_is_inside_current_repository": (
            REPO_ROOT in EXTERNAL_INPUT_PATH.resolve().parents
        ),
        "input_directly_matches_current_runner_contract": direct_runner_compatible,
        "recorded_metric_n_matches_external_input_rows": metric_n_matches,
        "conclusion": (
            "LEGACY_ARTIFACTS_NOT_RECONCILED"
            if not (direct_runner_compatible and metric_n_matches)
            else "RECONCILIATION_NOT_YET_SUFFICIENT_FOR_SCIENTIFIC_VALIDITY"
        ),
    }

    print(json.dumps(status, indent=2, ensure_ascii=False))

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import subprocess


# =============================================================================
# READ-ONLY EXTERNAL INPUT GENERATION-PATH DISCOVERY
# =============================================================================

CURRENT_REPO_ROOT = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

EXTERNAL_REPO_ROOT = Path("/content/cognitive-self-model").resolve()

EXTERNAL_INPUT_PATH = (
    EXTERNAL_REPO_ROOT
    / "reports"
    / "M21_2_4_3_4"
    / "input_dataset.csv"
)

SEARCH_TERMS = [
    "input_dataset.csv",
    "q_invalid",
    "self_model_invalid",
    "raw_q_invalid",
    "label_invalid",
    "episode_id",
    "split_role",
    "to_csv",
    "M21_2_4_3_4",
    "M21.2.4.3.4",
]

TEXT_SUFFIXES = {".py", ".ipynb", ".json", ".md", ".txt"}
MAX_FILE_BYTES = 2_000_000
CONTEXT_RADIUS = 2


def heading(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def run_command(
    command: list[str],
    cwd: Path | None = None,
) -> tuple[int, str, str]:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_probably_text(path: Path) -> bool:
    return path.suffix.lower() in TEXT_SUFFIXES and path.stat().st_size <= MAX_FILE_BYTES


def read_text_safely(path: Path) -> str | None:
    try:
        return path.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return None


def print_matching_contexts(path: Path, text: str) -> None:
    """
    Prints source context only; never prints CSV rows.
    """
    lines = text.splitlines()
    matched_lines = {
        index
        for index, line in enumerate(lines, start=1)
        if any(term.lower() in line.lower() for term in SEARCH_TERMS)
    }

    if not matched_lines:
        return

    relative = path.relative_to(EXTERNAL_REPO_ROOT)
    print(f"\n--- {relative} ---")

    visible_lines: set[int] = set()
    for line_number in matched_lines:
        visible_lines.update(
            range(
                max(1, line_number - CONTEXT_RADIUS),
                min(len(lines), line_number + CONTEXT_RADIUS) + 1,
            )
        )

    for line_number in sorted(visible_lines):
        marker = ">>" if line_number in matched_lines else "  "
        print(f"{marker} {line_number:05d}: {lines[line_number - 1]}")


# -----------------------------------------------------------------------------
# 1. External location and input file identity
# -----------------------------------------------------------------------------

heading("1. EXTERNAL LOCATION AND INPUT FILE IDENTITY")

print(f"External root: {EXTERNAL_REPO_ROOT}")
print(f"External root exists: {EXTERNAL_REPO_ROOT.exists()}")
print(f"Current audited repository: {CURRENT_REPO_ROOT}")
print(f"Input path: {EXTERNAL_INPUT_PATH}")
print(f"Input exists: {EXTERNAL_INPUT_PATH.exists()}")

if EXTERNAL_INPUT_PATH.exists():
    stat = EXTERNAL_INPUT_PATH.stat()
    print(f"Input size bytes: {stat.st_size}")
    print(f"Input modified UTC timestamp: {stat.st_mtime_ns}")
    print(f"Input SHA256: {sha256_file(EXTERNAL_INPUT_PATH)}")
else:
    print("STOP: external input CSV is absent.")
    raise FileNotFoundError(EXTERNAL_INPUT_PATH)

# -----------------------------------------------------------------------------
# 2. External Git identity, if this is a Git worktree
# -----------------------------------------------------------------------------

heading("2. EXTERNAL GIT IDENTITY")

git_dir = EXTERNAL_REPO_ROOT / ".git"
print(f"External .git exists: {git_dir.exists()}")

if git_dir.exists():
    git_queries = [
        ("origin", ["git", "remote", "get-url", "origin"]),
        ("branch", ["git", "branch", "--show-current"]),
        (
            "head",
            [
                "git",
                "log",
                "-1",
                "--format=%H%n%ad%n%s",
                "--date=iso-strict",
            ],
        ),
        ("status", ["git", "status", "--short"]),
    ]

    for label, command in git_queries:
        code, stdout, stderr = run_command(command, cwd=EXTERNAL_REPO_ROOT)
        value = stdout if code == 0 else f"<unavailable: {stderr}>"
        if label == "status" and code == 0 and not value:
            value = "<clean>"
        print(f"{label}: {value}")
else:
    print(
        "External directory is not a Git worktree; Git-level provenance cannot be "
        "established from this location."
    )

# -----------------------------------------------------------------------------
# 3. Git state/history of exact input path
# -----------------------------------------------------------------------------

heading("3. EXTERNAL INPUT GIT STATE AND HISTORY")

if git_dir.exists():
    relative_input = str(EXTERNAL_INPUT_PATH.relative_to(EXTERNAL_REPO_ROOT))

    checks = [
        (
            "Tracked status",
            ["git", "ls-files", "--error-unmatch", "--", relative_input],
        ),
        (
            "Ignore rule",
            ["git", "check-ignore", "-v", "--", relative_input],
        ),
        (
            "Path history",
            [
                "git",
                "log",
                "--follow",
                "--format=%H | %ad | %s",
                "--date=iso-strict",
                "-n",
                "20",
                "--",
                relative_input,
            ],
        ),
    ]

    for label, command in checks:
        print(f"\n{label}:")
        code, stdout, stderr = run_command(command, cwd=EXTERNAL_REPO_ROOT)
        print(stdout if code == 0 and stdout else f"<none or unavailable: {stderr}>")

# -----------------------------------------------------------------------------
# 4. Candidate generator discovery, excluding raw data files
# -----------------------------------------------------------------------------

heading("4. CANDIDATE GENERATOR FILE DISCOVERY")

excluded_directory_names = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "data",
    "artifacts",
}

candidate_files: list[Path] = []

for path in EXTERNAL_REPO_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if any(part in excluded_directory_names for part in path.relative_to(EXTERNAL_REPO_ROOT).parts):
        continue

    if path == EXTERNAL_INPUT_PATH:
        continue

    try:
        if not is_probably_text(path):
            continue
    except OSError:
        continue

    text = read_text_safely(path)
    if text is None:
        continue

    if any(term.lower() in text.lower() for term in SEARCH_TERMS):
        candidate_files.append(path)

print(f"Candidate text files found: {len(candidate_files)}")

for path in sorted(candidate_files):
    text = read_text_safely(path)
    if text is not None:
        print_matching_contexts(path, text)

# -----------------------------------------------------------------------------
# 5. Git content-search history for column names and export destination
# -----------------------------------------------------------------------------

heading("5. EXTERNAL GIT STRING-HISTORY SEARCH")

if git_dir.exists():
    for search_term in [
        "self_model_invalid",
        "q_invalid",
        "input_dataset.csv",
        "raw_q_invalid",
        "label_invalid",
    ]:
        print(f"\n--- git log -S {search_term!r} ---")

        code, stdout, stderr = run_command(
            [
                "git",
                "log",
                "-S",
                search_term,
                "--all",
                "--format=%H | %ad | %s",
                "--date=iso-strict",
                "-n",
                "20",
            ],
            cwd=EXTERNAL_REPO_ROOT,
        )

        print(stdout if code == 0 and stdout else f"<no matching history: {stderr}>")

# -----------------------------------------------------------------------------
# 6. Required provenance decision
# -----------------------------------------------------------------------------

heading("6. PROVENANCE DECISION RULE")

print(
    "A valid generation path requires evidence of all four mappings:\n"
    "  1. source artifact / generator -> episode_id\n"
    "  2. source estimator output -> q_invalid\n"
    "  3. authoritative observed evaluation label -> self_model_invalid\n"
    "  4. declared audit metadata -> track\n\n"
    "The output must also establish that q_invalid was computed without forbidden "
    "inputs, and that self_model_invalid is an evaluation target rather than an "
    "estimator/calibration input. Until then, the CSV remains an unproven local input."
)

In [ ]:
from __future__ import annotations

from pathlib import Path
import ast
import csv
import hashlib
import json
import re
import subprocess
from collections import Counter

import pandas as pd


# =============================================================================
# READ-ONLY M21.2.4.3.4 STAGE-2 CODE + ARTIFACT RECONCILIATION AUDIT
# =============================================================================

CURRENT_REPO_ROOT = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

EXTERNAL_ROOT = Path("/content/cognitive-self-model").resolve()

CURRENT_RUNNER = (
    CURRENT_REPO_ROOT
    / "src"
    / "legacy_import"
    / "m21_2_4_3_4_identifiability_benchmark.py"
)

EXTERNAL_RUNNER = (
    EXTERNAL_ROOT
    / "src"
    / "legacy_import"
    / "m21_2_4_3_4_identifiability_benchmark.py"
)

CURRENT_CONFIG = CURRENT_REPO_ROOT / "configs" / "m21_2_4_3_4_config.json"

EXTERNAL_REPORT_DIR = EXTERNAL_ROOT / "reports" / "M21_2_4_3_4"
EXTERNAL_INPUT = EXTERNAL_REPORT_DIR / "input_dataset.csv"
EXTERNAL_MANIFEST = EXTERNAL_REPORT_DIR / "manifest.json"
EXTERNAL_AUDIT = EXTERNAL_REPORT_DIR / "m21_2_4_3_4_identifiability_audit.json"
EXTERNAL_VERDICT = EXTERNAL_REPORT_DIR / "scientific_verdict.json"
EXTERNAL_FOLD_METRICS = EXTERNAL_REPORT_DIR / "fold_metrics.csv"
EXTERNAL_RELIABILITY = EXTERNAL_REPORT_DIR / "reliability_by_method.csv"
EXTERNAL_PERTURBATION = EXTERNAL_REPORT_DIR / "perturbation_stress.csv"
EXTERNAL_NEGATIVE_CONTROL = EXTERNAL_REPORT_DIR / "negative_control_results.json"
EXTERNAL_IDENTITY_ABLATION = EXTERNAL_REPORT_DIR / "sample_identity_ablation.json"

FORBIDDEN_INPUTS = {
    "groundtruth",
    "track",
    "true_mechanism",
    "future_observation",
    "governance_state",
    "decision_state",
}

EXPECTED_EXTERNAL_ARTIFACTS = [
    EXTERNAL_INPUT,
    EXTERNAL_MANIFEST,
    EXTERNAL_AUDIT,
    EXTERNAL_VERDICT,
    EXTERNAL_FOLD_METRICS,
    EXTERNAL_RELIABILITY,
    EXTERNAL_PERTURBATION,
    EXTERNAL_NEGATIVE_CONTROL,
    EXTERNAL_IDENTITY_ABLATION,
]


def heading(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_command(command: list[str], cwd: Path | None = None) -> tuple[int, str, str]:
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


def safe_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as exc:
        return {"_parse_error": str(exc)}


def source_line_count(path: Path) -> int | None:
    if not path.exists():
        return None
    return len(path.read_text(encoding="utf-8", errors="replace").splitlines())


def extract_python_string_literals(source: str) -> set[str]:
    """
    Extracts literals only for static review.
    This does not prove runtime data-flow compliance.
    """
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return set()

    strings: set[str] = set()

    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            strings.add(node.value)

    return strings


def find_source_hits(source: str, terms: set[str], radius: int = 1) -> list[str]:
    """
    Returns minimal code contexts. It never reads or prints CSV data.
    """
    lines = source.splitlines()
    hits: list[str] = []

    for index, line in enumerate(lines, start=1):
        lowered = line.lower()
        if any(term in lowered for term in terms):
            start = max(1, index - radius)
            end = min(len(lines), index + radius)
            context = "\n".join(
                f"{'>>' if line_no == index else '  '} {line_no:05d}: {lines[line_no - 1]}"
                for line_no in range(start, end + 1)
            )
            hits.append(context)

    return hits


def print_json_summary(label: str, payload) -> None:
    print(f"\n--- {label} ---")

    if payload is None:
        print("<missing>")
        return

    if isinstance(payload, dict) and "_parse_error" in payload:
        print(payload)
        return

    if isinstance(payload, dict):
        print(f"Top-level keys: {sorted(payload.keys())}")

        for key in [
            "benchmark",
            "milestone",
            "scientific_intent",
            "verdict",
            "scientific_verdict",
            "input_csv",
            "output_dir",
            "n_rows",
            "n_episodes",
            "seed_text",
            "n_folds",
            "rationale",
        ]:
            if key in payload:
                print(f"{key}: {payload[key]}")
    else:
        print(f"Top-level type: {type(payload).__name__}")


# -----------------------------------------------------------------------------
# 1. File existence and byte-level identity
# -----------------------------------------------------------------------------

heading("1. EXTERNAL STAGE-2 ARTIFACT INVENTORY")

for path in EXPECTED_EXTERNAL_ARTIFACTS:
    exists = path.exists()
    print(f"{path.relative_to(EXTERNAL_ROOT)}")
    print(f"  exists: {exists}")

    if exists:
        print(f"  bytes: {path.stat().st_size}")
        print(f"  sha256: {sha256_file(path)}")

# -----------------------------------------------------------------------------
# 2. Current tracked runner versus external stage-2 runner
# -----------------------------------------------------------------------------

heading("2. RUNNER IDENTITY AND CODE DIFFERENCE")

for label, path in [
    ("Current tracked runner", CURRENT_RUNNER),
    ("External stage-2 runner", EXTERNAL_RUNNER),
]:
    print(f"\n{label}: {path}")
    print(f"Exists: {path.exists()}")

    if path.exists():
        print(f"Source lines: {source_line_count(path)}")
        print(f"SHA256: {sha256_file(path)}")

if CURRENT_RUNNER.exists() and EXTERNAL_RUNNER.exists():
    current_hash = sha256_file(CURRENT_RUNNER)
    external_hash = sha256_file(EXTERNAL_RUNNER)

    print(f"\nByte-identical: {current_hash == external_hash}")

    code, stdout, stderr = run_command(
        [
            "diff",
            "-u",
            "--label",
            "tracked_runner",
            str(CURRENT_RUNNER),
            "--label",
            "external_stage2_runner",
            str(EXTERNAL_RUNNER),
        ]
    )

    diff_lines = stdout.splitlines()

    # A bounded diff avoids flooding the notebook.
    print(f"Unified diff exit code: {code} (0 means identical; 1 means different)")
    print(f"Unified diff total lines: {len(diff_lines)}")

    print("\nFirst 160 diff lines:")
    for line in diff_lines[:160]:
        print(line)

    if len(diff_lines) > 160:
        print(f"... {len(diff_lines) - 160} additional diff lines omitted.")

# -----------------------------------------------------------------------------
# 3. External source static-contract review
# -----------------------------------------------------------------------------

heading("3. EXTERNAL RUNNER STATIC CONTRACT REVIEW")

if not EXTERNAL_RUNNER.exists():
    raise FileNotFoundError(f"Missing external runner: {EXTERNAL_RUNNER}")

external_source = EXTERNAL_RUNNER.read_text(encoding="utf-8", errors="replace")
string_literals = extract_python_string_literals(external_source)
lowered_literals = {value.lower() for value in string_literals}

expected_contract_names = {
    "episode_id",
    "q_invalid",
    "self_model_invalid",
    "track",
}

print("Expected external CSV contract names:")
for name in sorted(expected_contract_names):
    print(f"  {name}: {name in string_literals}")

print("\nForbidden-token presence in source literals:")
for forbidden in sorted(FORBIDDEN_INPUTS):
    print(f"  {forbidden}: {forbidden in lowered_literals}")

print(
    "\nInterpretation rule: static token absence is supportive but not a proof of "
    "runtime non-use. Static token presence must be reviewed in context."
)

print("\nContexts containing score/label/episode/track/split/calibration terms:")
review_terms = {
    "episode_id",
    "q_invalid",
    "self_model_invalid",
    "track",
    "split",
    "fold",
    "calibr",
    "train",
    "eval",
    "bootstrap",
    "sha256",
}

contexts = find_source_hits(external_source, review_terms, radius=1)

for context in contexts[:120]:
    print(context)
    print()

if len(contexts) > 120:
    print(f"... {len(contexts) - 120} additional source contexts omitted.")

# -----------------------------------------------------------------------------
# 4. AST-level check: calls that may construct split assignments
# -----------------------------------------------------------------------------

heading("4. SPLIT-PROTOCOL STATIC LEADS")

try:
    tree = ast.parse(external_source)
except SyntaxError as exc:
    raise RuntimeError(f"External runner has Python syntax error: {exc}") from exc

candidate_calls = []

for node in ast.walk(tree):
    if not isinstance(node, ast.Call):
        continue

    try:
        expression = ast.unparse(node.func)
    except Exception:
        expression = "<unparseable>"

    expression_lower = expression.lower()

    if any(
        token in expression_lower
        for token in [
            "kfold",
            "split",
            "shuffle",
            "sample",
            "group",
            "bootstrap",
            "choice",
            "permutation",
        ]
    ):
        candidate_calls.append((node.lineno, expression))

for line_number, expression in sorted(set(candidate_calls)):
    print(f"Line {line_number}: call target = {expression}")

print(
    "\nRequired manual conclusion after reviewing contexts:\n"
    "  - Are train/calibration/evaluation episode sets constructed from episode_id?\n"
    "  - Is their intersection explicitly checked to be empty?\n"
    "  - Is track excluded from estimator/calibrator inputs and used only for reporting?\n"
    "  - Are all random choices derived from a stable SHA256-based seed path?"
)

# -----------------------------------------------------------------------------
# 5. Privacy-safe summaries of outputs
# -----------------------------------------------------------------------------

heading("5. EXTERNAL OUTPUT SUMMARIES")

manifest = safe_json(EXTERNAL_MANIFEST)
audit = safe_json(EXTERNAL_AUDIT)
verdict = safe_json(EXTERNAL_VERDICT)
negative_control = safe_json(EXTERNAL_NEGATIVE_CONTROL)
identity_ablation = safe_json(EXTERNAL_IDENTITY_ABLATION)

print_json_summary("manifest.json", manifest)
print_json_summary("m21_2_4_3_4_identifiability_audit.json", audit)
print_json_summary("scientific_verdict.json", verdict)
print_json_summary("negative_control_results.json", negative_control)
print_json_summary("sample_identity_ablation.json", identity_ablation)

for label, path in [
    ("fold_metrics.csv", EXTERNAL_FOLD_METRICS),
    ("reliability_by_method.csv", EXTERNAL_RELIABILITY),
    ("perturbation_stress.csv", EXTERNAL_PERTURBATION),
]:
    print(f"\n--- {label} ---")

    if not path.exists():
        print("<missing>")
        continue

    frame = pd.read_csv(path)
    print(f"Rows: {len(frame)}")
    print(f"Columns: {list(frame.columns)}")
    print(f"Missing counts: {frame.isna().sum().astype(int).to_dict()}")

    # Only aggregate/count information; no row-level output.
    for column in [
        "method",
        "fold",
        "track",
        "seed",
        "split_role",
        "calibration_method",
    ]:
        if column in frame.columns:
            print(f"{column} counts: {frame[column].value_counts(dropna=False).to_dict()}")

    numeric_columns = frame.select_dtypes(include="number").columns.tolist()
    if numeric_columns:
        numeric_summary = {
            column: {
                "min": float(frame[column].min()),
                "max": float(frame[column].max()),
                "mean": float(frame[column].mean()),
            }
            for column in numeric_columns
        }
        print(f"Numeric summaries: {numeric_summary}")

# -----------------------------------------------------------------------------
# 6. Input-to-manifest consistency checks
# -----------------------------------------------------------------------------

heading("6. INPUT-TO-MANIFEST CONSISTENCY")

if not EXTERNAL_INPUT.exists():
    raise FileNotFoundError(EXTERNAL_INPUT)

input_df = pd.read_csv(EXTERNAL_INPUT)

observed_input_summary = {
    "n_rows": int(len(input_df)),
    "n_episodes": int(input_df["episode_id"].nunique(dropna=True))
    if "episode_id" in input_df.columns
    else None,
    "columns": list(input_df.columns),
}

print(f"Observed input summary: {observed_input_summary}")

if isinstance(manifest, dict):
    for key in ["n_rows", "n_episodes"]:
        recorded = manifest.get(key)
        observed = observed_input_summary.get(key)
        print(f"Manifest {key}: {recorded}")
        print(f"Observed {key}: {observed}")
        print(f"Match: {recorded == observed}")

    print(f"Manifest input_csv: {manifest.get('input_csv')}")
    print(f"Exact manifest input path matches: {manifest.get('input_csv') == str(EXTERNAL_INPUT)}")

# -----------------------------------------------------------------------------
# 7. Final status
# -----------------------------------------------------------------------------

heading("7. RECONCILIATION STATUS")

external_runner_exists = EXTERNAL_RUNNER.exists()
external_input_exists = EXTERNAL_INPUT.exists()
external_git_worktree = (EXTERNAL_ROOT / ".git").exists()

manifest_counts_match = False
if isinstance(manifest, dict):
    manifest_counts_match = (
        manifest.get("n_rows") == observed_input_summary["n_rows"]
        and manifest.get("n_episodes") == observed_input_summary["n_episodes"]
    )

result = {
    "external_stage2_runner_exists": external_runner_exists,
    "external_input_exists": external_input_exists,
    "external_input_schema": observed_input_summary["columns"],
    "external_manifest_counts_match_input": manifest_counts_match,
    "external_location_is_git_worktree": external_git_worktree,
    "tracked_and_external_runner_byte_identical": (
        CURRENT_RUNNER.exists()
        and EXTERNAL_RUNNER.exists()
        and sha256_file(CURRENT_RUNNER) == sha256_file(EXTERNAL_RUNNER)
    ),
    "scientific_interpretation": (
        "EXTERNAL_RUN_HAS_COMPUTATIONAL_ARTIFACTS_BUT_NOT_YET_AUTHORITATIVE_PROVENANCE"
    ),
}

print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
{
  "benchmark": "M21.2.4.3.4",
  "input_sha256": "52fd6d5d771a086cfbdc7939bbef56d4d6a08dc04b83147a820f23af9467f2d4",
  "input_schema": [
    "episode_id",
    "track",
    "q_invalid",
    "self_model_invalid"
  ],
  "n_rows": 1200,
  "n_episodes": 60,
  "score_column": "q_invalid",
  "label_column": "self_model_invalid",
  "episode_column": "episode_id",
  "track_column": "track",
  "score_semantics": "support_aware_invalidity_evidence_score",
  "label_semantics": "observed_evaluation_target_for_self_model_invalidity",
  "estimator_forbidden_inputs": [
    "GroundTruth",
    "Track",
    "true_mechanism",
    "future_observation",
    "governance_state",
    "decision_state"
  ],
  "track_usage": "post_hoc_stratified_audit_only",
  "generator_provenance_status": "NOT_YET_ESTABLISHED"
}

In [ ]:
from __future__ import annotations

from pathlib import Path
import subprocess


# =====================================================================
# Repository URL: https://github.com/YOUR_USERNAME/cognitive-self-model
# Default branch: main
# Current milestone: TO_BE_VERIFIED_FROM_REPOSITORY
#
# Read-only repository inspection.
# No file modification, no git staging, no commit, no push.
# =====================================================================

REPO_DIR = Path("/content/cognitive-self-model")
DEFAULT_BRANCH = "main"


def run_command(command: list[str], cwd: Path) -> tuple[int, str]:
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return completed.returncode, completed.stdout.strip()


def print_file_preview(path: Path, max_lines: int = 160) -> None:
    print(f"\n--- {path.relative_to(REPO_DIR)} ---")
    if not path.exists():
        print("NOT FOUND")
        return

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        print("NOT DISPLAYED: file is not valid UTF-8 text.")
        return

    for line in lines[:max_lines]:
        print(line)

    if len(lines) > max_lines:
        print(f"\n... truncated: showing {max_lines} of {len(lines)} lines ...")


def print_tree(path: Path, max_depth: int = 3) -> None:
    print(f"\n--- Directory tree: {path.relative_to(REPO_DIR) if path != REPO_DIR else '.'} ---")

    if not path.exists():
        print("NOT FOUND")
        return

    ignored_names = {
        ".git",
        "__pycache__",
        ".ipynb_checkpoints",
        ".pytest_cache",
        ".mypy_cache",
        ".venv",
        "venv",
    }

    def walk(current: Path, depth: int) -> None:
        if depth > max_depth:
            return

        try:
            children = sorted(
                [
                    item for item in current.iterdir()
                    if item.name not in ignored_names
                ],
                key=lambda item: (not item.is_dir(), item.name.lower()),
            )
        except PermissionError:
            print("  " * depth + "[Permission denied]")
            return

        for item in children:
            prefix = "  " * depth
            suffix = "/" if item.is_dir() else ""
            print(f"{prefix}{item.name}{suffix}")
            if item.is_dir():
                walk(item, depth + 1)

    walk(path, depth=0)


if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository path not found: {REPO_DIR}\n"
        "Set REPO_DIR to the local directory of your cloned private repository."
    )

if not (REPO_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_DIR} is not recognized as a Git repository because '.git' was not found."
    )

print("=" * 88)
print("COGNITIVE-SELF-MODEL — READ-ONLY REPOSITORY INSPECTION")
print("=" * 88)

print("\n[1] Repository path")
print(REPO_DIR.resolve())

print("\n[2] Git remote metadata")
_, output = run_command(["git", "remote", "-v"], REPO_DIR)
print(output or "No Git remote configured.")

print("\n[3] Current branch and Git status")
_, branch = run_command(["git", "branch", "--show-current"], REPO_DIR)
_, status = run_command(["git", "status", "-sb"], REPO_DIR)
print(f"Current branch: {branch or '<detached HEAD>'}")
print(status or "No git status output.")

print("\n[4] Default branch availability")
_, local_main = run_command(["git", "branch", "--list", DEFAULT_BRANCH], REPO_DIR)
_, remote_main = run_command(
    ["git", "branch", "-r", "--list", f"origin/{DEFAULT_BRANCH}"],
    REPO_DIR,
)
print(f"Local '{DEFAULT_BRANCH}': {'FOUND' if local_main else 'NOT FOUND'}")
print(f"Remote 'origin/{DEFAULT_BRANCH}': {'FOUND' if remote_main else 'NOT FOUND'}")

print("\n[5] Latest commits")
_, commits = run_command(
    ["git", "log", "--decorate", "--oneline", "-12"],
    REPO_DIR,
)
print(commits or "No commits found.")

print_tree(REPO_DIR, max_depth=2)

for directory_name in [
    "src",
    "notebooks",
    "reports",
    "artifacts",
    "configs",
    "tests",
    "data",
]:
    print_tree(REPO_DIR / directory_name, max_depth=3)

print("\n[6] Repository guidance and setup files")
for relative_path in [
    "README.md",
    "AGENTS.md",
    "AI_PROJECT_INSTRUCTIONS.md",
    "pyproject.toml",
    "requirements.txt",
    "environment.yml",
]:
    print_file_preview(REPO_DIR / relative_path, max_lines=180)

print("\n[7] Candidate files related to M21 / calibration / invalidity")
keywords = (
    "m21",
    "calibrat",
    "invalidity",
    "isotonic",
    "pava",
    "self_model",
)

candidates: list[Path] = []
for path in REPO_DIR.rglob("*"):
    if not path.is_file():
        continue
    if ".git" in path.parts or "__pycache__" in path.parts:
        continue

    relative_lower = str(path.relative_to(REPO_DIR)).lower()
    if any(keyword in relative_lower for keyword in keywords):
        candidates.append(path)

if not candidates:
    print("No candidate files found by filename.")
else:
    for path in sorted(candidates, key=lambda item: str(item).lower()):
        print(path.relative_to(REPO_DIR))

print("\n[8] Recent reports and configurations previews")
for base_dir_name in ["reports", "configs", "artifacts"]:
    base_dir = REPO_DIR / base_dir_name
    if not base_dir.exists():
        continue

    selected = sorted(
        [
            path for path in base_dir.rglob("*")
            if path.is_file()
            and path.suffix.lower() in {".md", ".json", ".txt", ".csv"}
            and any(
                keyword in str(path.relative_to(REPO_DIR)).lower()
                for keyword in keywords
            )
        ],
        key=lambda item: item.stat().st_mtime,
        reverse=True,
    )[:8]

    for path in selected:
        print_file_preview(path, max_lines=100)

print("\n" + "=" * 88)
print("INSPECTION COMPLETE")
print("No repository files were changed.")
print("No files were staged.")
print("No commit or push was performed.")
print("=" * 88)

In [ ]:
from pathlib import Path
import os


# =====================================================================
# Repository URL: https://github.com/YOUR_USERNAME/cognitive-self-model
# Default branch: main
# Current milestone: TO_BE_VERIFIED_FROM_REPOSITORY
#
# Read-only local path inspection.
# No clone, no git init, no file modification.
# =====================================================================

SEARCH_ROOT = Path("/content")
MAX_DEPTH = 4

print("=" * 88)
print("LOCAL REPOSITORY PATH INSPECTION")
print("=" * 88)

print(f"\nCurrent working directory:\n{Path.cwd()}")

print("\nTop-level entries in /content:")
if SEARCH_ROOT.exists():
    for item in sorted(SEARCH_ROOT.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower())):
        suffix = "/" if item.is_dir() else ""
        print(f"  {item.name}{suffix}")
else:
    print("/content does not exist.")

print("\nGit repositories found under /content:")
git_dirs = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    current = Path(root)

    try:
        relative_depth = len(current.relative_to(SEARCH_ROOT).parts)
    except ValueError:
        relative_depth = MAX_DEPTH + 1

    if relative_depth > MAX_DEPTH:
        dirs[:] = []
        continue

    ignored = {".cache", ".config", "__pycache__", ".ipynb_checkpoints"}
    dirs[:] = [name for name in dirs if name not in ignored]

    if ".git" in dirs:
        git_dirs.append(current)
        dirs.remove(".git")

if git_dirs:
    for repo_path in sorted(git_dirs):
        print(f"  FOUND: {repo_path}")
else:
    print("  No '.git' directory was found under /content.")

print("\nDirectories whose names may be related to this project:")
keywords = ("cognitive", "self", "model", "repo", "github")
matching_dirs = []

for root, dirs, _ in os.walk(SEARCH_ROOT):
    current = Path(root)

    try:
        relative_depth = len(current.relative_to(SEARCH_ROOT).parts)
    except ValueError:
        relative_depth = MAX_DEPTH + 1

    if relative_depth > MAX_DEPTH:
        dirs[:] = []
        continue

    if current != SEARCH_ROOT and any(keyword in current.name.lower() for keyword in keywords):
        matching_dirs.append(current)

for path in sorted(set(matching_dirs)):
    print(f"  {path}")

print("\nExpected-path diagnostic:")
expected_path = Path("/content/cognitive-self-model")
print(f"  Path exists: {(expected_path.exists())}")
print(f"  Is directory: {(expected_path.is_dir())}")
print(f"  Contains .git: {(expected_path / '.git').exists()}")

if expected_path.is_dir():
    print("\nContents of /content/cognitive-self-model:")
    for item in sorted(expected_path.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower())):
        suffix = "/" if item.is_dir() else ""
        print(f"  {item.name}{suffix}")

print("\n" + "=" * 88)
print("INSPECTION COMPLETE — NOTHING WAS MODIFIED")
print("=" * 88)

In [ ]:
from pathlib import Path
import subprocess


# =====================================================================
# Repository URL: https://github.com/YOUR_USERNAME/cognitive-self-model
# Default branch: main
# Current milestone: TO_BE_VERIFIED_FROM_REPOSITORY
#
# Read-only repository inspection.
# No file modification, no staging, no commit, no push.
# =====================================================================

REPO_DIR = Path("/content/cognitive-self-model-scientific-structure")


def run(command: list[str]) -> str:
    result = subprocess.run(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return result.stdout.strip()


def show_text_file(relative_path: str, max_lines: int = 180) -> None:
    path = REPO_DIR / relative_path
    print(f"\n{'=' * 88}\nFILE: {relative_path}\n{'=' * 88}")

    if not path.exists():
        print("NOT FOUND")
        return

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        print("Cannot preview: non-UTF-8 file.")
        return

    print("\n".join(lines[:max_lines]))
    if len(lines) > max_lines:
        print(f"\n... truncated: {max_lines}/{len(lines)} lines shown ...")


print("=" * 88)
print("COGNITIVE-SELF-MODEL — AUTHORITATIVE REPOSITORY INSPECTION")
print("=" * 88)

print(f"\nRepository path: {REPO_DIR}")
print(f"Has .git: {(REPO_DIR / '.git').exists()}")

print("\n[1] Git remote")
print(run(["git", "remote", "-v"]) or "No remote configured.")

print("\n[2] Current branch and status")
print("Current branch:", run(["git", "branch", "--show-current"]) or "<detached HEAD>")
print(run(["git", "status", "-sb"]) or "No status output.")

print("\n[3] Branches")
print(run(["git", "branch", "-a"]) or "No branches found.")

print("\n[4] Latest commits")
print(run(["git", "log", "--decorate", "--oneline", "-12"]) or "No commits found.")

print("\n[5] Top-level repository files/directories")
for item in sorted(REPO_DIR.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower())):
    if item.name == ".git":
        continue
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

for directory_name in [
    "src",
    "notebooks",
    "reports",
    "artifacts",
    "configs",
    "tests",
    "data",
]:
    directory = REPO_DIR / directory_name
    print(f"\n[6] {directory_name}/")
    if not directory.exists():
        print("NOT FOUND")
        continue

    files = sorted(
        [
            str(path.relative_to(REPO_DIR))
            for path in directory.rglob("*")
            if path.is_file()
            and ".ipynb_checkpoints" not in path.parts
            and "__pycache__" not in path.parts
        ]
    )

    if not files:
        print("EMPTY")
    else:
        for relative_path in files[:100]:
            print(f"  {relative_path}")

        if len(files) > 100:
            print(f"  ... truncated: 100/{len(files)} files shown ...")

show_text_file("README.md")
show_text_file("AGENTS.md")
show_text_file("AI_PROJECT_INSTRUCTIONS.md")

print("\n" + "=" * 88)
print("INSPECTION COMPLETE — NO FILES WERE MODIFIED")
print("=" * 88)

In [ ]:
from pathlib import Path
import json
import subprocess


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main (not currently present in this local clone)
# Current branch: chore/repository-scientific-structure
# Inspection scope: M21.2.4.3.3 and M21.2.4.3.4
#
# Read-only inspection:
# - no file modification
# - no git add
# - no commit
# - no push
# - no dataset generation
# =====================================================================

REPO_DIR = Path("/content/cognitive-self-model-scientific-structure")

TARGET_FILES = [
    ".gitignore",
    "requirements.txt",
    "configs/m21_2_4_3_3_config.json",
    "configs/m21_2_4_3_4_config.json",
    "reports/M21_2_4_3_3/scientific_verdict.json",
    "reports/M21_2_4_3_3/calibration_support_summary.json",
    "reports/M21_2_4_3_3/exact_output_audit.json",
    "reports/M21_2_4_3_4/scientific_verdict.json",
    "reports/M21_2_4_3_4/m21_2_4_3_4_identifiability_audit.json",
    "reports/M21_2_4_3_4/fold_metrics.csv",
    "src/common/reproducibility.py",
    "src/common/io_contracts.py",
    "src/legacy_import/m21_2_4_3_4_identifiability_benchmark.py",
]


def run_git(command: list[str]) -> str:
    result = subprocess.run(
        command,
        cwd=REPO_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    return result.stdout.strip()


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def preview_text(relative_path: str, max_lines: int = 260) -> None:
    path = REPO_DIR / relative_path
    print_header(f"FILE: {relative_path}")

    if not path.exists():
        print("NOT FOUND")
        return

    try:
        text = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        print("Cannot preview: non-UTF-8 file.")
        return

    lines = text.splitlines()

    if relative_path.endswith(".json"):
        try:
            parsed = json.loads(text)
            print(json.dumps(parsed, indent=2, ensure_ascii=False, sort_keys=True))
            return
        except json.JSONDecodeError:
            print("WARNING: File has .json extension but could not be parsed as JSON.")
            print()

    print("\n".join(lines[:max_lines]))

    if len(lines) > max_lines:
        print(f"\n... OUTPUT TRUNCATED: showing {max_lines} of {len(lines)} lines ...")


print_header("M21.2.4.3.3 / M21.2.4.3.4 — READ-ONLY SCIENTIFIC INSPECTION")

print(f"Repository: {REPO_DIR}")
print(f"Branch: {run_git(['git', 'branch', '--show-current'])}")
print(f"HEAD: {run_git(['git', 'rev-parse', '--short', 'HEAD'])}")
print("\nGit status:")
print(run_git(["git", "status", "-sb"]) or "<no output>")

for relative_path in TARGET_FILES:
    preview_text(relative_path)

print_header("INSPECTION COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from pathlib import Path
import json
import re


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main
# Current branch: chore/repository-scientific-structure
# Current milestone under inspection: M21.2.4.3.4
#
# Read-only inspection only:
# - no writes
# - no git add
# - no commit
# - no push
# - no benchmark execution
# =====================================================================

REPO_DIR = Path("/content/cognitive-self-model-scientific-structure")

FILES_TO_PREVIEW = [
    "src/legacy_import/m21_2_4_3_1_pipeline.py",
    "src/legacy_import/m21_2_4_3_3_audit.py",
    "src/legacy_import/m212433_calibration_audit.py",
    "notebooks/cognitive_self_model_latest.ipynb",
    "tests/test_episode_id_contract.py",
    "tests/test_npz_contract.py",
]


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def preview_text_file(relative_path: str, max_lines: int = 320) -> None:
    path = REPO_DIR / relative_path
    print_header(f"FILE: {relative_path}")

    if not path.exists():
        print("NOT FOUND")
        return

    try:
        lines = path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        print("Cannot preview non-UTF-8 text.")
        return

    print("\n".join(lines[:max_lines]))

    if len(lines) > max_lines:
        print(f"\n... TRUNCATED: showing {max_lines} of {len(lines)} lines ...")


def preview_notebook_code(relative_path: str, max_cells: int = 80) -> None:
    path = REPO_DIR / relative_path
    print_header(f"NOTEBOOK CODE CELLS: {relative_path}")

    if not path.exists():
        print("NOT FOUND")
        return

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not parse notebook JSON: {exc}")
        return

    cells = notebook.get("cells", [])
    code_cells = [
        cell for cell in cells
        if cell.get("cell_type") == "code"
    ]

    print(f"Notebook cells: {len(cells)}")
    print(f"Code cells: {len(code_cells)}")

    for index, cell in enumerate(code_cells[:max_cells], start=1):
        source = "".join(cell.get("source", []))
        print("\n" + "-" * 100)
        print(f"CODE CELL {index}")
        print("-" * 100)
        print(source.rstrip() or "<empty code cell>")

    if len(code_cells) > max_cells:
        print(f"\n... TRUNCATED: showing {max_cells} of {len(code_cells)} code cells ...")


def search_repository(pattern: str, allowed_suffixes: set[str]) -> None:
    print_header(f"SEARCH: {pattern!r}")

    compiled = re.compile(pattern, flags=re.IGNORECASE)
    matches = []

    for path in REPO_DIR.rglob("*"):
        if (
            not path.is_file()
            or ".git" in path.parts
            or "__pycache__" in path.parts
            or path.suffix.lower() not in allowed_suffixes
        ):
            continue

        try:
            lines = path.read_text(encoding="utf-8").splitlines()
        except (UnicodeDecodeError, OSError):
            continue

        for line_number, line in enumerate(lines, start=1):
            if compiled.search(line):
                matches.append(
                    (
                        str(path.relative_to(REPO_DIR)),
                        line_number,
                        line.strip(),
                    )
                )

    if not matches:
        print("No matches found.")
        return

    for relative_path, line_number, line in matches[:250]:
        print(f"{relative_path}:{line_number}: {line}")

    if len(matches) > 250:
        print(f"\n... TRUNCATED: showing 250 of {len(matches)} matches ...")


for file_path in FILES_TO_PREVIEW:
    if file_path.endswith(".ipynb"):
        preview_notebook_code(file_path)
    else:
        preview_text_file(file_path)

TEXT_SUFFIXES = {".py", ".md", ".json", ".txt", ".ipynb"}

search_repository(
    pattern=r"input_dataset|raw_q_invalid|label_invalid|split_role|episode_id",
    allowed_suffixes=TEXT_SUFFIXES,
)

search_repository(
    pattern=r"M21\.?2\.?4\.?3\.?4|M21_2_4_3_4",
    allowed_suffixes=TEXT_SUFFIXES,
)

print_header("INSPECTION COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Current branch: chore/repository-scientific-structure
# Milestone: M21.2.4.3.4
#
# Purpose:
# Read-only, privacy-preserving audit of the local input CSV contract.
#
# Safety:
# - no writes
# - no generated files
# - no git commands that modify state
# - no row-level private data printed
# - no raw text or payload columns printed
# =====================================================================

REPO_DIR = Path("/content/cognitive-self-model-scientific-structure")
INPUT_CSV = REPO_DIR / "reports/M21_2_4_3_4/input_dataset.csv"

REQUIRED_COLUMNS = (
    "episode_id",
    "split_role",
    "raw_q_invalid",
    "label_invalid",
    "track",
)

ALLOWED_SPLIT_ROLES = {
    "train",
    "calibration",
    "evaluation",
    "test",
}


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


print_header("M21.2.4.3.4 — PRIVATE INPUT CSV CONTRACT AUDIT")

print(f"Repository directory: {REPO_DIR}")
print(f"Input CSV path: {INPUT_CSV}")
print(f"Input CSV exists: {INPUT_CSV.exists()}")

if not INPUT_CSV.exists():
    print(
        "\nSTOP: Local input CSV was not found at the expected ignored path.\n"
        "No benchmark implementation or rerun should proceed until the "
        "private input artifact location is confirmed."
    )

else:
    print(f"Input CSV size (bytes): {INPUT_CSV.stat().st_size}")
    print(f"Input CSV SHA256: {sha256_file(INPUT_CSV)}")

    dataframe = pd.read_csv(INPUT_CSV, low_memory=False)

    print_header("SCHEMA")

    print(f"Number of rows: {len(dataframe)}")
    print(f"Number of columns: {len(dataframe.columns)}")
    print("Columns:")
    for column in dataframe.columns:
        print(f"  - {column}: {dataframe[column].dtype}")

    missing_columns = sorted(
        set(REQUIRED_COLUMNS) - set(dataframe.columns)
    )
    unexpected_required_check = sorted(
        set(dataframe.columns) - set(REQUIRED_COLUMNS)
    )

    print("\nRequired-column validation:")
    print(f"  Missing required columns: {missing_columns}")
    print(f"  Additional columns: {unexpected_required_check}")

    if missing_columns:
        raise ValueError(
            "Input CSV contract failure. "
            f"Missing required columns: {missing_columns}"
        )

    print_header("NULL / TYPE / VALUE VALIDATION")

    null_counts = dataframe.loc[:, REQUIRED_COLUMNS].isna().sum()
    print("Null counts in required columns:")
    print(null_counts.to_string())

    score = pd.to_numeric(
        dataframe["raw_q_invalid"],
        errors="coerce",
    )
    label = pd.to_numeric(
        dataframe["label_invalid"],
        errors="coerce",
    )

    print("\nraw_q_invalid:")
    print(f"  Non-numeric count: {int(score.isna().sum())}")
    print(f"  Non-finite count: {int((~np.isfinite(score.fillna(np.nan))).sum())}")
    print(f"  Minimum: {float(score.min())}")
    print(f"  Maximum: {float(score.max())}")
    print(f"  Outside [0, 1] count: {int(((score < 0) | (score > 1)).sum())}")

    observed_labels = sorted(
        pd.Series(label.dropna().unique()).astype(float).tolist()
    )
    invalid_label_count = int(
        (~label.isin([0, 1])).sum()
    )

    print("\nlabel_invalid:")
    print(f"  Observed values: {observed_labels}")
    print(f"  Values outside {{0, 1}}: {invalid_label_count}")
    print(f"  Prevalence: {float(label.mean())}")

    print_header("EPISODE IDENTITY AND SPLIT-LEAKAGE AUDIT")

    episode_id = dataframe["episode_id"].astype(str)
    split_role = dataframe["split_role"].astype(str)

    episode_summary = (
        pd.DataFrame({
            "episode_id": episode_id,
            "split_role": split_role,
        })
        .drop_duplicates()
        .groupby("episode_id", as_index=False)
        .agg(
            n_distinct_split_roles=(
                "split_role",
                "nunique",
            ),
            split_roles=(
                "split_role",
                lambda values: "|".join(sorted(set(values))),
            ),
        )
    )

    globally_unique_episode_count = int(episode_id.nunique())
    repeated_row_episode_count = int(
        (episode_id.value_counts() > 1).sum()
    )
    cross_role_episode_count = int(
        (episode_summary["n_distinct_split_roles"] > 1).sum()
    )

    print(f"Distinct episode_id count: {globally_unique_episode_count}")
    print(f"Episodes represented by multiple rows: {repeated_row_episode_count}")
    print(
        "Episodes appearing in more than one split_role: "
        f"{cross_role_episode_count}"
    )

    if cross_role_episode_count:
        print(
            "\nCONTRACT FAILURE: An episode occurs in multiple split roles. "
            "This violates episode-disjoint evaluation."
        )
    else:
        print(
            "\nPASS: No episode_id appears in more than one split_role."
        )

    print_header("SPLIT COMPOSITION — ROW AND EPISODE LEVEL")

    split_summary = (
        dataframe.assign(
            _episode_id=episode_id,
            _label=label,
        )
        .groupby("split_role", dropna=False)
        .agg(
            n_rows=("_episode_id", "size"),
            n_episodes=("_episode_id", "nunique"),
            prevalence=("_label", "mean"),
        )
        .sort_index()
    )
    print(split_summary.to_string())

    observed_split_roles = set(split_role.unique())
    unknown_split_roles = sorted(
        observed_split_roles - ALLOWED_SPLIT_ROLES
    )

    print("\nObserved split roles:")
    print(sorted(observed_split_roles))
    print(f"Unexpected split roles: {unknown_split_roles}")

    print_header("TRACK COMPOSITION — DESCRIPTIVE ONLY")

    track_summary = (
        dataframe.assign(
            _episode_id=episode_id,
            _label=label,
        )
        .groupby("track", dropna=False)
        .agg(
            n_rows=("_episode_id", "size"),
            n_episodes=("_episode_id", "nunique"),
            prevalence=("_label", "mean"),
        )
        .sort_index()
    )
    print(track_summary.to_string())

    print_header("SCORE SUPPORT DIAGNOSTICS")

    finite_score = score[np.isfinite(score)]
    score_counts = finite_score.value_counts(dropna=False)

    print(f"Distinct raw_q_invalid values: {int(finite_score.nunique())}")
    print(
        "Fraction of score values occurring once: "
        f"{float((score_counts == 1).mean()) if len(score_counts) else float('nan')}"
    )
    print(f"Exact raw_q_invalid == 0 count: {int((finite_score == 0.0).sum())}")
    print(f"Exact raw_q_invalid == 1 count: {int((finite_score == 1.0).sum())}")

    quantiles = finite_score.quantile(
        [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]
    )
    print("\nraw_q_invalid quantiles:")
    print(quantiles.to_string())

    print_header("AUDIT SUMMARY")

    summary = {
        "input_csv_exists": True,
        "input_csv_sha256": sha256_file(INPUT_CSV),
        "n_rows": int(len(dataframe)),
        "n_columns": int(len(dataframe.columns)),
        "missing_required_columns": missing_columns,
        "null_counts_required_columns": {
            key: int(value)
            for key, value in null_counts.to_dict().items()
        },
        "n_distinct_episode_ids": globally_unique_episode_count,
        "episodes_in_multiple_split_roles": cross_role_episode_count,
        "observed_split_roles": sorted(observed_split_roles),
        "unknown_split_roles": unknown_split_roles,
        "raw_score_min": float(score.min()),
        "raw_score_max": float(score.max()),
        "raw_score_outside_unit_interval_count": int(
            ((score < 0) | (score > 1)).sum()
        ),
        "label_values": observed_labels,
        "invalid_label_count": invalid_label_count,
    }

    print(json.dumps(summary, indent=2, ensure_ascii=False))

print_header("AUDIT COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib


# =====================================================================
# M21.2.4.3.4 — READ-ONLY PRIVATE ARTIFACT LOCATION AUDIT
#
# Purpose:
# Locate possible private input_dataset.csv artifacts in common Colab
# locations without printing, loading, copying, or modifying their data.
#
# Safety:
# - no writes
# - no file copies or moves
# - no Git changes
# - no CSV contents printed
# =====================================================================

SEARCH_ROOTS = [
    Path("/content/cognitive-self-model-scientific-structure"),
    Path("/content/cognitive-self-model"),
    Path("/content/drive/MyDrive"),
]

TARGET_FILENAMES = {
    "input_dataset.csv",
}

MAX_RESULTS = 100


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


print_header("M21.2.4.3.4 — PRIVATE INPUT ARTIFACT LOCATION AUDIT")

matches: list[Path] = []

for root in SEARCH_ROOTS:
    print(f"\nSearch root: {root}")
    print(f"Exists: {root.exists()}")

    if not root.exists():
        continue

    try:
        root_matches = [
            path
            for path in root.rglob("*")
            if path.is_file() and path.name in TARGET_FILENAMES
        ]
    except PermissionError as exc:
        print(f"Permission denied while searching: {exc}")
        continue

    print(f"Matching filename count: {len(root_matches)}")
    matches.extend(root_matches)

unique_matches = sorted(set(matches))

print_header("MATCHES")

if not unique_matches:
    print(
        "No input_dataset.csv file was found in the inspected roots.\n"
        "The private artifact is unavailable in this runtime, or it has a "
        "different filename/location."
    )
else:
    for index, path in enumerate(unique_matches[:MAX_RESULTS], start=1):
        print(f"\n[{index}] Path: {path}")
        print(f"    Size (bytes): {path.stat().st_size}")
        print(f"    SHA256: {sha256_file(path)}")

    if len(unique_matches) > MAX_RESULTS:
        print(
            f"\n... truncated: showing {MAX_RESULTS} of "
            f"{len(unique_matches)} matching files ..."
        )

print_header("AUDIT COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

from pathlib import Path
import re


# =====================================================================
# M21.2.4.3.4 — REPOSITORY CONTRACT-EVIDENCE DISCOVERY
#
# Purpose:
# Read repository code/config/docs to determine the intended input schema
# and whether split roles are input-provided or benchmark-derived.
#
# Safety:
# - read-only
# - no private CSV loading
# - no file writes
# - no Git changes
# =====================================================================

REPOSITORY_DIR = Path(
    "/content/cognitive-self-model-scientific-structure"
)

MILESTONE_TOKENS = (
    "M21.2.4.3.4",
    "M21_2_4_3_4",
    "m21_2_4_3_4",
)

COLUMN_TOKENS = (
    "q_invalid",
    "self_model_invalid",
    "raw_q_invalid",
    "label_invalid",
    "split_role",
    "episode_id",
    "track",
)

SPLIT_TOKENS = (
    "cross-fitting",
    "cross_fitting",
    "episode-disjoint",
    "episode_disjoint",
    "calibration",
    "evaluation",
    "fold",
    "split",
    "sha256",
)

ALLOWED_SUFFIXES = {
    ".py",
    ".md",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".txt",
}

EXCLUDED_DIR_NAMES = {
    ".git",
    "__pycache__",
    ".ipynb_checkpoints",
    "data",
    "artifacts",
}

MAX_FILE_BYTES = 2_000_000
CONTEXT_LINES = 2


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def is_candidate_file(path: Path) -> bool:
    if path.suffix.lower() not in ALLOWED_SUFFIXES:
        return False

    if any(parent.name in EXCLUDED_DIR_NAMES for parent in path.parents):
        return False

    try:
        return path.stat().st_size <= MAX_FILE_BYTES
    except OSError:
        return False


def contains_relevant_token(text: str) -> bool:
    text_lower = text.lower()

    return any(
        token.lower() in text_lower
        for token in (*MILESTONE_TOKENS, *COLUMN_TOKENS)
    )


def print_matching_context(
    path: Path,
    lines: list[str],
) -> int:
    match_indexes: set[int] = set()

    for index, line in enumerate(lines):
        line_lower = line.lower()

        if any(
            token.lower() in line_lower
            for token in (*MILESTONE_TOKENS, *COLUMN_TOKENS, *SPLIT_TOKENS)
        ):
            match_indexes.add(index)

    if not match_indexes:
        return 0

    print(f"\nFILE: {path.relative_to(REPOSITORY_DIR)}")

    printed_indexes: set[int] = set()

    for match_index in sorted(match_indexes):
        start_index = max(0, match_index - CONTEXT_LINES)
        end_index = min(
            len(lines),
            match_index + CONTEXT_LINES + 1,
        )

        for line_index in range(start_index, end_index):
            if line_index in printed_indexes:
                continue

            print(
                f"{line_index + 1:>5}: "
                f"{lines[line_index].rstrip()}"
            )
            printed_indexes.add(line_index)

        print("-----")

    return 1


print_header(
    "M21.2.4.3.4 — REPOSITORY CONTRACT-EVIDENCE DISCOVERY"
)

print(f"Repository directory: {REPOSITORY_DIR}")
print(f"Repository exists: {REPOSITORY_DIR.exists()}")

if not REPOSITORY_DIR.exists():
    raise FileNotFoundError(
        f"Repository directory not found: {REPOSITORY_DIR}"
    )

candidate_paths = sorted(
    path
    for path in REPOSITORY_DIR.rglob("*")
    if path.is_file() and is_candidate_file(path)
)

print(f"Read-only candidate source/config/doc files: {len(candidate_paths)}")

print_header("RELEVANT FILES AND CONTEXT")

relevant_file_count = 0
read_error_count = 0

for path in candidate_paths:
    try:
        text = path.read_text(
            encoding="utf-8",
            errors="replace",
        )
    except OSError as exc:
        read_error_count += 1
        print(f"\nREAD ERROR: {path} -> {exc}")
        continue

    if not contains_relevant_token(text):
        continue

    lines = text.splitlines()

    relevant_file_count += print_matching_context(
        path=path,
        lines=lines,
    )

print_header("EXPLICIT CONFIG PATH CHECK")

expected_config_paths = (
    REPOSITORY_DIR / "configs/m21_2_4_3_4_config.json",
    REPOSITORY_DIR / "configs/M21_2_4_3_4_config.json",
)

for config_path in expected_config_paths:
    print(f"\nPath: {config_path}")
    print(f"Exists: {config_path.exists()}")

    if config_path.exists():
        print("Content:")
        print(
            config_path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        )

print_header("OBSERVED PRIVATE ARTIFACT SCHEMA — METADATA ONLY")

print("Confirmed artifact path:")
print(
    "/content/cognitive-self-model/"
    "reports/M21_2_4_3_4/input_dataset.csv"
)
print("Confirmed SHA256:")
print("52fd6d5d771a086cfbdc7939bbef56d4d6a08dc04b83147a820f23af9467f2d4")
print("Observed columns from prior read-only audit:")
print("  - episode_id")
print("  - track")
print("  - q_invalid")
print("  - self_model_invalid")

print_header("DISCOVERY COMPLETE — NO FILES WERE MODIFIED")

print(f"Relevant files with printed context: {relevant_file_count}")
print(f"Read errors: {read_error_count}")

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main
# Current milestone: M21.2.4.3.4
#
# Purpose:
# Read-only feasibility audit for the private base artifact.
#
# Important contract:
# - q_invalid is the raw score used by calibrators.
# - self_model_invalid is the binary calibration/evaluation outcome.
# - track is audit metadata and part of global episode identity only.
# - track is NOT an estimator/calibrator input or split stratification key.
#
# Safety:
# - no writes
# - no copy / move / delete
# - no generated split assignment
# - no Git changes
# - no row-level private data printed
# =====================================================================

INPUT_CSV = Path(
    "/content/cognitive-self-model/"
    "reports/M21_2_4_3_4/input_dataset.csv"
)

CONFIG_JSON = Path(
    "/content/cognitive-self-model-scientific-structure/"
    "configs/m21_2_4_3_4_config.json"
)

EXPECTED_SHA256 = (
    "52fd6d5d771a086cfbdc7939bbef56d4d6a08dc04b83147a820f23af9467f2d4"
)

REQUIRED_INPUT_COLUMNS = (
    "episode_id",
    "track",
    "q_invalid",
    "self_model_invalid",
)


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def make_global_episode_key(
    track: pd.Series,
    episode_id: pd.Series,
) -> pd.Series:
    """
    Protocol identity only.

    This prevents collisions when the same local episode_id appears
    under different tracks. It must not be passed to a calibrator.
    """
    return (
        track.astype(str)
        + "::"
        + episode_id.astype(str)
    )


print_header(
    "M21.2.4.3.4 — BASE ARTIFACT EPISODE FEASIBILITY AUDIT"
)

print(f"Input CSV: {INPUT_CSV}")
print(f"Input exists: {INPUT_CSV.exists()}")
print(f"Config JSON: {CONFIG_JSON}")
print(f"Config exists: {CONFIG_JSON.exists()}")

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Private input artifact not found: {INPUT_CSV}"
    )

if not CONFIG_JSON.exists():
    raise FileNotFoundError(
        f"Milestone config not found: {CONFIG_JSON}"
    )

observed_sha256 = sha256_file(INPUT_CSV)

print(f"Expected SHA256: {EXPECTED_SHA256}")
print(f"Observed SHA256: {observed_sha256}")
print(f"SHA256 match: {observed_sha256 == EXPECTED_SHA256}")

if observed_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Private artifact SHA256 mismatch. Stop before analysis."
    )

with CONFIG_JSON.open("r", encoding="utf-8") as file_handle:
    config = json.load(file_handle)

episode_protocol = config["episode_protocol"]

train_min_episodes = int(
    episode_protocol["train_min_episodes"]
)
calibration_min_episodes = int(
    episode_protocol["calibration_min_episodes"]
)
calibration_preferred_episodes = int(
    episode_protocol["calibration_preferred_episodes"]
)
eval_min_episodes = int(
    episode_protocol["eval_min_episodes"]
)

minimum_total_required_episodes = (
    train_min_episodes
    + calibration_min_episodes
    + eval_min_episodes
)

preferred_total_required_episodes = (
    train_min_episodes
    + calibration_preferred_episodes
    + eval_min_episodes
)

dataframe = pd.read_csv(INPUT_CSV, low_memory=False)

print_header("INPUT CONTRACT VALIDATION")

print(f"Number of rows: {int(len(dataframe))}")
print(f"Observed columns: {sorted(dataframe.columns.tolist())}")

missing_columns = sorted(
    set(REQUIRED_INPUT_COLUMNS) - set(dataframe.columns)
)

additional_columns = sorted(
    set(dataframe.columns) - set(REQUIRED_INPUT_COLUMNS)
)

print(f"Missing required columns: {missing_columns}")
print(f"Additional columns: {additional_columns}")

if missing_columns:
    raise ValueError(
        "Base artifact schema failure. "
        f"Missing required columns: {missing_columns}"
    )

required_null_counts = (
    dataframe.loc[:, REQUIRED_INPUT_COLUMNS]
    .isna()
    .sum()
)

print("\nNull counts in required columns:")
print(required_null_counts.to_string())

if int(required_null_counts.sum()) > 0:
    raise ValueError(
        "Base artifact contains null values in required columns."
    )

score = pd.to_numeric(
    dataframe["q_invalid"],
    errors="coerce",
)

label = pd.to_numeric(
    dataframe["self_model_invalid"],
    errors="coerce",
)

if score.isna().any():
    raise ValueError(
        "q_invalid contains non-numeric values."
    )

if label.isna().any():
    raise ValueError(
        "self_model_invalid contains non-numeric values."
    )

finite_score_mask = np.isfinite(score.to_numpy())

if not finite_score_mask.all():
    raise ValueError(
        "q_invalid contains non-finite values."
    )

score_outside_unit_interval = (
    (score < 0.0)
    | (score > 1.0)
)

observed_label_values = sorted(
    label.astype(float).unique().tolist()
)

non_binary_label_count = int(
    (~label.isin([0, 1])).sum()
)

print("\nScore and label validity:")
print(f"q_invalid minimum: {float(score.min())}")
print(f"q_invalid maximum: {float(score.max())}")
print(
    "q_invalid values outside [0, 1]: "
    f"{int(score_outside_unit_interval.sum())}"
)
print(f"self_model_invalid observed values: {observed_label_values}")
print(
    "self_model_invalid values outside {{0, 1}}: "
    f"{non_binary_label_count}"
)
print(
    "Overall invalidity prevalence: "
    f"{float(label.mean())}"
)

if int(score_outside_unit_interval.sum()) > 0:
    raise ValueError(
        "q_invalid must lie within [0, 1]."
    )

if non_binary_label_count > 0:
    raise ValueError(
        "self_model_invalid must be binary (0/1)."
    )

print_header("GLOBAL EPISODE IDENTITY AUDIT")

local_episode_id = dataframe["episode_id"].astype(str)
track = dataframe["track"].astype(str)

global_episode_key = make_global_episode_key(
    track=track,
    episode_id=local_episode_id,
)

local_episode_track_counts = (
    pd.DataFrame(
        {
            "episode_id": local_episode_id,
            "track": track,
        }
    )
    .drop_duplicates()
    .groupby("episode_id", dropna=False)["track"]
    .nunique()
)

local_ids_reused_across_tracks = int(
    (local_episode_track_counts > 1).sum()
)

episode_row_counts = global_episode_key.value_counts(
    dropna=False
)

n_rows = int(len(dataframe))
n_tracks = int(track.nunique(dropna=False))
n_local_episode_ids = int(
    local_episode_id.nunique(dropna=False)
)
n_global_episodes = int(
    global_episode_key.nunique(dropna=False)
)

print(f"Distinct tracks: {n_tracks}")
print(f"Distinct local episode_id values: {n_local_episode_ids}")
print(f"Distinct global episode keys: {n_global_episodes}")
print(
    "Local episode_id values reused across tracks: "
    f"{local_ids_reused_across_tracks}"
)
print(
    "Episodes represented by multiple rows: "
    f"{int((episode_row_counts > 1).sum())}"
)
print(
    "Rows per global episode — min / median / max: "
    f"{int(episode_row_counts.min())} / "
    f"{float(episode_row_counts.median())} / "
    f"{int(episode_row_counts.max())}"
)

print_header("EPISODE-LEVEL LABEL VARIATION")

episode_label_summary = (
    pd.DataFrame(
        {
            "global_episode_key": global_episode_key,
            "label": label.astype(int),
        }
    )
    .groupby("global_episode_key", as_index=False)
    .agg(
        n_rows=("label", "size"),
        n_distinct_labels=("label", "nunique"),
        label_prevalence=("label", "mean"),
    )
)

episodes_with_mixed_labels = int(
    (episode_label_summary["n_distinct_labels"] > 1).sum()
)

print(
    "Episodes containing both label classes across rows: "
    f"{episodes_with_mixed_labels}"
)
print(
    "Episode-level label-prevalence range: "
    f"{float(episode_label_summary['label_prevalence'].min())} .. "
    f"{float(episode_label_summary['label_prevalence'].max())}"
)

print_header("TRACK COMPOSITION — DESCRIPTIVE AUDIT ONLY")

track_summary = (
    pd.DataFrame(
        {
            "track": track,
            "global_episode_key": global_episode_key,
            "label": label.astype(int),
        }
    )
    .groupby("track", dropna=False)
    .agg(
        n_rows=("global_episode_key", "size"),
        n_episodes=("global_episode_key", "nunique"),
        invalidity_prevalence=("label", "mean"),
    )
    .sort_index()
)

print(
    track_summary.to_string(
        float_format=lambda value: f"{value:.6f}"
    )
)

print_header("CONFIGURED EPISODE-PROTOCOL FEASIBILITY")

print(f"Train minimum episodes: {train_min_episodes}")
print(f"Calibration minimum episodes: {calibration_min_episodes}")
print(
    "Calibration preferred episodes: "
    f"{calibration_preferred_episodes}"
)
print(f"Evaluation minimum episodes: {eval_min_episodes}")
print(
    "Episode-disjoint required: "
    f"{episode_protocol['episode_disjoint_required']}"
)

print(
    "\nMinimum total disjoint episodes required "
    "(train + calibration-min + evaluation): "
    f"{minimum_total_required_episodes}"
)
print(
    "Preferred total disjoint episodes required "
    "(train + calibration-preferred + evaluation): "
    f"{preferred_total_required_episodes}"
)
print(
    "Available global episodes: "
    f"{n_global_episodes}"
)

minimum_protocol_feasible = (
    n_global_episodes >= minimum_total_required_episodes
)

preferred_protocol_feasible = (
    n_global_episodes >= preferred_total_required_episodes
)

print(
    "\nMinimum protocol feasibility: "
    f"{'PASS' if minimum_protocol_feasible else 'FAIL'}"
)
print(
    "Preferred protocol feasibility: "
    f"{'PASS' if preferred_protocol_feasible else 'NOT_MET'}"
)

if not minimum_protocol_feasible:
    print(
        "\nSCIENTIFIC STOP CONDITION:"
        "\nThe artifact does not contain enough globally distinct "
        "episodes for the configured disjoint train/calibration/"
        "evaluation protocol. No calibration benchmark split should "
        "be constructed from this artifact under the current config."
    )
else:
    print(
        "\nProtocol capacity is sufficient for a later deterministic "
        "episode-disjoint split audit. This cell intentionally does "
        "not create, assign, or save any split."
    )

print_header("COMPACT READ-ONLY SUMMARY")

summary = {
    "milestone": config["milestone"],
    "artifact_sha256": observed_sha256,
    "artifact_sha256_match": observed_sha256 == EXPECTED_SHA256,
    "n_rows": n_rows,
    "n_tracks": n_tracks,
    "n_local_episode_ids": n_local_episode_ids,
    "n_global_episodes": n_global_episodes,
    "local_episode_ids_reused_across_tracks": (
        local_ids_reused_across_tracks
    ),
    "episodes_with_mixed_row_labels": episodes_with_mixed_labels,
    "q_invalid_min": float(score.min()),
    "q_invalid_max": float(score.max()),
    "q_invalid_outside_unit_interval_count": int(
        score_outside_unit_interval.sum()
    ),
    "label_values": observed_label_values,
    "overall_invalidity_prevalence": float(label.mean()),
    "train_min_episodes": train_min_episodes,
    "calibration_min_episodes": calibration_min_episodes,
    "calibration_preferred_episodes": (
        calibration_preferred_episodes
    ),
    "eval_min_episodes": eval_min_episodes,
    "minimum_total_required_episodes": (
        minimum_total_required_episodes
    ),
    "preferred_total_required_episodes": (
        preferred_total_required_episodes
    ),
    "minimum_protocol_feasible": minimum_protocol_feasible,
    "preferred_protocol_feasible": preferred_protocol_feasible,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

print_header("AUDIT COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main
# Current milestone: M21.2.4.3.4
#
# Purpose:
# Read-only deterministic episode-disjoint split protocol audit.
#
# Important:
# - q_invalid is the calibration input score.
# - self_model_invalid is the binary outcome.
# - track is used only to construct global episode identity:
#       global_episode_key = track + "::" + episode_id
# - track is NOT used as a calibrator feature, estimator feature,
#   model-selection variable, or split-stratification variable.
# - No calibration method is fitted in this cell.
#
# Safety:
# - no writes
# - no artifact generation
# - no private row-level data printed
# - no Git operations
# =====================================================================

INPUT_CSV = Path(
    "/content/cognitive-self-model/"
    "reports/M21_2_4_3_4/input_dataset.csv"
)

CONFIG_JSON = Path(
    "/content/cognitive-self-model-scientific-structure/"
    "configs/m21_2_4_3_4_config.json"
)

EXPECTED_SHA256 = (
    "52fd6d5d771a086cfbdc7939bbef56d4d6a08dc04b83147a820f23af9467f2d4"
)


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def make_global_episode_key(
    track: pd.Series,
    episode_id: pd.Series,
) -> pd.Series:
    """
    Global identity only.

    track is never passed to an estimator or calibrator.
    """
    return track.astype(str) + "::" + episode_id.astype(str)


def stable_episode_rank(
    global_episode_key: str,
    namespace: str,
    seed_value: int,
) -> str:
    """
    Stable, platform-independent rank token.

    The key includes only:
      - fixed milestone namespace,
      - seed value,
      - already-constructed global episode identity.

    No q_invalid, label, row order, track-specific split policy,
    or future information affects assignment.
    """
    payload = (
        f"{namespace}"
        f"|seed={int(seed_value)}"
        f"|episode={global_episode_key}"
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


def build_role_assignment(
    episode_keys: list[str],
    namespace: str,
    seed_value: int,
    calibration_episodes: int,
    evaluation_episodes: int,
) -> pd.DataFrame:
    """
    Assign roles by deterministic SHA256 rank.

    Role semantics:
      - calibration: calibrator fitting only
      - evaluation: unseen evaluation only
      - train_reserved: retained only to honor the configured protocol;
                        it must not be consumed by any calibrator,
                        estimator, threshold selection, or verdict rule.
    """
    assignment = pd.DataFrame(
        {
            "global_episode_key": sorted(episode_keys),
        }
    )

    assignment["stable_rank"] = assignment[
        "global_episode_key"
    ].map(
        lambda key: stable_episode_rank(
            global_episode_key=key,
            namespace=namespace,
            seed_value=seed_value,
        )
    )

    assignment = assignment.sort_values(
        by=["stable_rank", "global_episode_key"],
        kind="mergesort",
    ).reset_index(drop=True)

    n_episodes = len(assignment)

    if calibration_episodes + evaluation_episodes > n_episodes:
        raise ValueError(
            "Insufficient global episodes for requested "
            "calibration/evaluation allocation."
        )

    assignment["split_role"] = "train_reserved"

    assignment.loc[
        : calibration_episodes - 1,
        "split_role",
    ] = "calibration"

    evaluation_start = calibration_episodes
    evaluation_end = (
        calibration_episodes
        + evaluation_episodes
    )

    assignment.loc[
        evaluation_start:evaluation_end - 1,
        "split_role",
    ] = "evaluation"

    return assignment.loc[
        :,
        [
            "global_episode_key",
            "split_role",
        ],
    ].copy()


print_header(
    "M21.2.4.3.4 — DETERMINISTIC EPISODE-DISJOINT SPLIT AUDIT"
)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Input not found: {INPUT_CSV}")

if not CONFIG_JSON.exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_JSON}")

observed_sha256 = sha256_file(INPUT_CSV)

print(f"Expected input SHA256: {EXPECTED_SHA256}")
print(f"Observed input SHA256: {observed_sha256}")
print(f"SHA256 match: {observed_sha256 == EXPECTED_SHA256}")

if observed_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "Input identity mismatch. Stop before split audit."
    )

with CONFIG_JSON.open("r", encoding="utf-8") as file_handle:
    config = json.load(file_handle)

protocol = config["episode_protocol"]
namespace = str(config["random_seed_namespace"])
seed_values = [int(seed) for seed in config["seed_values"]]

calibration_episode_count = int(
    protocol["calibration_preferred_episodes"]
)
evaluation_episode_count = int(
    protocol["eval_min_episodes"]
)
train_min_episodes = int(
    protocol["train_min_episodes"]
)

if not bool(protocol["episode_disjoint_required"]):
    raise ValueError(
        "This audit requires episode_disjoint_required=true."
    )

dataframe = pd.read_csv(INPUT_CSV, low_memory=False)

global_episode_key = make_global_episode_key(
    track=dataframe["track"],
    episode_id=dataframe["episode_id"],
)

audit_dataframe = pd.DataFrame(
    {
        "global_episode_key": global_episode_key,
        "label": dataframe["self_model_invalid"].astype(int),
    }
)

episode_keys = sorted(
    audit_dataframe["global_episode_key"]
    .unique()
    .tolist()
)

n_global_episodes = len(episode_keys)

required_episodes = (
    train_min_episodes
    + calibration_episode_count
    + evaluation_episode_count
)

print_header("SPLIT-PROTOCOL PARAMETERS")

print(f"Random seed namespace: {namespace}")
print(f"Configured seed values: {seed_values}")
print(
    "Calibration episodes per seed "
    f"(preferred): {calibration_episode_count}"
)
print(
    "Evaluation episodes per seed: "
    f"{evaluation_episode_count}"
)
print(
    "Minimum train-reserved episodes: "
    f"{train_min_episodes}"
)
print(f"Available global episodes: {n_global_episodes}")
print(
    "Required global episodes for configured roles: "
    f"{required_episodes}"
)

if n_global_episodes < required_episodes:
    raise RuntimeError(
        "Configured episode protocol is infeasible. "
        f"Available={n_global_episodes}, "
        f"required={required_episodes}."
    )

print(
    "Protocol capacity status: PASS "
    "(no split has been persisted)."
)

print_header("SEED-WISE DISJOINTNESS AND AGGREGATE COMPOSITION")

seed_summary_rows: list[dict[str, object]] = []

for seed_value in seed_values:
    assignment_first = build_role_assignment(
        episode_keys=episode_keys,
        namespace=namespace,
        seed_value=seed_value,
        calibration_episodes=calibration_episode_count,
        evaluation_episodes=evaluation_episode_count,
    )

    assignment_second = build_role_assignment(
        episode_keys=episode_keys,
        namespace=namespace,
        seed_value=seed_value,
        calibration_episodes=calibration_episode_count,
        evaluation_episodes=evaluation_episode_count,
    )

    deterministic_replay_pass = assignment_first.equals(
        assignment_second
    )

    role_counts = assignment_first[
        "split_role"
    ].value_counts()

    calibration_keys = set(
        assignment_first.loc[
            assignment_first["split_role"] == "calibration",
            "global_episode_key",
        ]
    )

    evaluation_keys = set(
        assignment_first.loc[
            assignment_first["split_role"] == "evaluation",
            "global_episode_key",
        ]
    )

    train_reserved_keys = set(
        assignment_first.loc[
            assignment_first["split_role"] == "train_reserved",
            "global_episode_key",
        ]
    )

    overlap_calibration_evaluation = (
        calibration_keys.intersection(evaluation_keys)
    )
    overlap_calibration_reserved = (
        calibration_keys.intersection(train_reserved_keys)
    )
    overlap_evaluation_reserved = (
        evaluation_keys.intersection(train_reserved_keys)
    )

    if (
        overlap_calibration_evaluation
        or overlap_calibration_reserved
        or overlap_evaluation_reserved
    ):
        raise AssertionError(
            f"Episode overlap detected for seed={seed_value}."
        )

    merged = audit_dataframe.merge(
        assignment_first,
        on="global_episode_key",
        how="left",
        validate="many_to_one",
    )

    if merged["split_role"].isna().any():
        raise AssertionError(
            f"Unassigned rows detected for seed={seed_value}."
        )

    aggregate = (
        merged.groupby("split_role", sort=True)
        .agg(
            n_rows=("label", "size"),
            n_episodes=("global_episode_key", "nunique"),
            invalidity_prevalence=("label", "mean"),
        )
        .reset_index()
    )

    aggregate_lookup = aggregate.set_index("split_role")

    for role in (
        "calibration",
        "evaluation",
        "train_reserved",
    ):
        if role not in aggregate_lookup.index:
            raise AssertionError(
                f"Missing role={role} for seed={seed_value}."
            )

    seed_summary_rows.append(
        {
            "seed": seed_value,
            "deterministic_replay_pass": deterministic_replay_pass,
            "calibration_n_episodes": int(
                aggregate_lookup.loc[
                    "calibration",
                    "n_episodes",
                ]
            ),
            "calibration_n_rows": int(
                aggregate_lookup.loc[
                    "calibration",
                    "n_rows",
                ]
            ),
            "calibration_invalidity_prevalence": float(
                aggregate_lookup.loc[
                    "calibration",
                    "invalidity_prevalence",
                ]
            ),
            "evaluation_n_episodes": int(
                aggregate_lookup.loc[
                    "evaluation",
                    "n_episodes",
                ]
            ),
            "evaluation_n_rows": int(
                aggregate_lookup.loc[
                    "evaluation",
                    "n_rows",
                ]
            ),
            "evaluation_invalidity_prevalence": float(
                aggregate_lookup.loc[
                    "evaluation",
                    "invalidity_prevalence",
                ]
            ),
            "train_reserved_n_episodes": int(
                aggregate_lookup.loc[
                    "train_reserved",
                    "n_episodes",
                ]
            ),
            "train_reserved_n_rows": int(
                aggregate_lookup.loc[
                    "train_reserved",
                    "n_rows",
                ]
            ),
            "calibration_evaluation_overlap": len(
                overlap_calibration_evaluation
            ),
            "calibration_train_reserved_overlap": len(
                overlap_calibration_reserved
            ),
            "evaluation_train_reserved_overlap": len(
                overlap_evaluation_reserved
            ),
        }
    )

seed_summary = pd.DataFrame(seed_summary_rows)

print(
    seed_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print_header("PROTOCOL ASSERTIONS")

all_replay_deterministic = bool(
    seed_summary["deterministic_replay_pass"].all()
)

all_calibration_sizes_correct = bool(
    (
        seed_summary["calibration_n_episodes"]
        == calibration_episode_count
    ).all()
)

all_evaluation_sizes_correct = bool(
    (
        seed_summary["evaluation_n_episodes"]
        == evaluation_episode_count
    ).all()
)

all_train_minimums_correct = bool(
    (
        seed_summary["train_reserved_n_episodes"]
        >= train_min_episodes
    ).all()
)

all_overlaps_zero = bool(
    (
        seed_summary[
            [
                "calibration_evaluation_overlap",
                "calibration_train_reserved_overlap",
                "evaluation_train_reserved_overlap",
            ]
        ]
        == 0
    )
    .all()
    .all()
)

assertion_summary = {
    "input_sha256_match": (
        observed_sha256 == EXPECTED_SHA256
    ),
    "n_seed_values_matches_config": (
        len(seed_values) == int(config["n_seeds"])
    ),
    "all_deterministic_replays_pass": all_replay_deterministic,
    "all_calibration_episode_counts_correct": (
        all_calibration_sizes_correct
    ),
    "all_evaluation_episode_counts_correct": (
        all_evaluation_sizes_correct
    ),
    "all_train_reserved_minimums_satisfied": (
        all_train_minimums_correct
    ),
    "all_role_overlaps_zero": all_overlaps_zero,
    "track_used_only_for_global_episode_identity": True,
    "calibrator_fitted_in_this_audit": False,
    "files_modified": False,
}

print(
    json.dumps(
        assertion_summary,
        indent=2,
        ensure_ascii=False,
    )
)

if not all(assertion_summary.values()):
    raise AssertionError(
        "At least one deterministic split-protocol assertion failed."
    )

print_header("AUDIT COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

import json

import pandas as pd


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main
# Current milestone: M21.2.4.3.4
#
# Read-only class-support audit.
#
# Preconditions:
# - dataframe
# - audit_dataframe
# - episode_keys
# - namespace
# - seed_values
# - calibration_episode_count
# - evaluation_episode_count
# - build_role_assignment(...)
#
# All variables should already exist from the deterministic split audit.
# =====================================================================


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


print_header(
    "M21.2.4.3.4 — SEED-WISE CLASS-SUPPORT AUDIT"
)

required_runtime_names = [
    "audit_dataframe",
    "episode_keys",
    "namespace",
    "seed_values",
    "calibration_episode_count",
    "evaluation_episode_count",
    "build_role_assignment",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Required variables are not available in the current runtime. "
        "Run the deterministic split audit cell first. "
        f"Missing: {missing_runtime_names}"
    )

support_rows: list[dict[str, object]] = []

for seed_value in seed_values:
    assignment = build_role_assignment(
        episode_keys=episode_keys,
        namespace=namespace,
        seed_value=seed_value,
        calibration_episodes=calibration_episode_count,
        evaluation_episodes=evaluation_episode_count,
    )

    merged = audit_dataframe.merge(
        assignment,
        on="global_episode_key",
        how="left",
        validate="many_to_one",
    )

    for split_role in (
        "calibration",
        "evaluation",
        "train_reserved",
    ):
        role_frame = merged.loc[
            merged["split_role"] == split_role
        ].copy()

        n_negative_rows = int(
            (role_frame["label"] == 0).sum()
        )
        n_positive_rows = int(
            (role_frame["label"] == 1).sum()
        )

        episode_class_summary = (
            role_frame.groupby(
                "global_episode_key",
                as_index=False,
            )["label"]
            .agg(
                episode_has_negative=lambda values: int(
                    (values == 0).any()
                ),
                episode_has_positive=lambda values: int(
                    (values == 1).any()
                ),
            )
        )

        n_episodes_with_negative = int(
            episode_class_summary[
                "episode_has_negative"
            ].sum()
        )
        n_episodes_with_positive = int(
            episode_class_summary[
                "episode_has_positive"
            ].sum()
        )

        support_rows.append(
            {
                "seed": int(seed_value),
                "split_role": split_role,
                "n_rows": int(len(role_frame)),
                "n_episodes": int(
                    role_frame["global_episode_key"].nunique()
                ),
                "n_negative_rows": n_negative_rows,
                "n_positive_rows": n_positive_rows,
                "n_episodes_with_negative_rows": (
                    n_episodes_with_negative
                ),
                "n_episodes_with_positive_rows": (
                    n_episodes_with_positive
                ),
                "both_row_classes_present": bool(
                    n_negative_rows > 0
                    and n_positive_rows > 0
                ),
                "both_classes_represented_across_episodes": bool(
                    n_episodes_with_negative > 0
                    and n_episodes_with_positive > 0
                ),
            }
        )

support_summary = pd.DataFrame(support_rows)

print(
    support_summary.to_string(
        index=False,
    )
)

calibration_support = support_summary.loc[
    support_summary["split_role"] == "calibration"
].copy()

evaluation_support = support_summary.loc[
    support_summary["split_role"] == "evaluation"
].copy()

assertions = {
    "all_calibration_splits_have_both_row_classes": bool(
        calibration_support[
            "both_row_classes_present"
        ].all()
    ),
    "all_evaluation_splits_have_both_row_classes": bool(
        evaluation_support[
            "both_row_classes_present"
        ].all()
    ),
    "all_calibration_splits_have_episode_class_representation": bool(
        calibration_support[
            "both_classes_represented_across_episodes"
        ].all()
    ),
    "all_evaluation_splits_have_episode_class_representation": bool(
        evaluation_support[
            "both_classes_represented_across_episodes"
        ].all()
    ),
    "files_modified": False,
    "calibrator_fitted": False,
}

print_header("CLASS-SUPPORT ASSERTIONS")

print(
    json.dumps(
        assertions,
        indent=2,
        ensure_ascii=False,
    )
)

if not assertions[
    "all_calibration_splits_have_both_row_classes"
]:
    raise AssertionError(
        "At least one calibration split lacks a label class; "
        "Platt scaling cannot be fitted for that seed."
    )

if not assertions[
    "all_evaluation_splits_have_both_row_classes"
]:
    raise AssertionError(
        "At least one evaluation split lacks a label class; "
        "some discrimination metrics would be undefined."
    )

if assertions["files_modified"]:
    raise AssertionError("Unexpected file modification.")

if assertions["calibrator_fitted"]:
    raise AssertionError(
        "Unexpected calibrator fitting in support audit."
    )

print_header("CLASS-SUPPORT AUDIT COMPLETE — NO FILES WERE MODIFIED")

In [ ]:
from __future__ import annotations

import json

import pandas as pd


# =====================================================================
# Repository URL: https://github.com/motiza345/cognitive-self-model.git
# Default branch: main
# Current milestone: M21.2.4.3.4
#
# Purpose:
# Read-only seed-wise track coverage audit.
#
# track is used only for descriptive reporting here.
# It is not passed to any calibrator or estimator.
#
# Preconditions:
# - dataframe
# - audit_dataframe
# - episode_keys
# - namespace
# - seed_values
# - calibration_episode_count
# - evaluation_episode_count
# - build_role_assignment(...)
# =====================================================================


def print_header(title: str) -> None:
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


required_runtime_names = [
    "dataframe",
    "audit_dataframe",
    "episode_keys",
    "namespace",
    "seed_values",
    "calibration_episode_count",
    "evaluation_episode_count",
    "build_role_assignment",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Run the deterministic split audit cell first. "
        f"Missing variables: {missing_runtime_names}"
    )

print_header(
    "M21.2.4.3.4 — SEED-WISE TRACK-COVERAGE AUDIT"
)

track_identity_frame = pd.DataFrame(
    {
        "global_episode_key": (
            dataframe["track"].astype(str)
            + "::"
            + dataframe["episode_id"].astype(str)
        ),
        "track": dataframe["track"].astype(str),
        "label": dataframe["self_model_invalid"].astype(int),
    }
)

expected_tracks = sorted(
    track_identity_frame["track"].unique().tolist()
)

coverage_rows: list[dict[str, object]] = []

for seed_value in seed_values:
    assignment = build_role_assignment(
        episode_keys=episode_keys,
        namespace=namespace,
        seed_value=seed_value,
        calibration_episodes=calibration_episode_count,
        evaluation_episodes=evaluation_episode_count,
    )

    merged = track_identity_frame.merge(
        assignment,
        on="global_episode_key",
        how="left",
        validate="many_to_one",
    )

    if merged["split_role"].isna().any():
        raise AssertionError(
            f"Unassigned rows found for seed={seed_value}."
        )

    for split_role in ("calibration", "evaluation"):
        role_frame = merged.loc[
            merged["split_role"] == split_role
        ].copy()

        role_summary = (
            role_frame.groupby("track", as_index=False)
            .agg(
                n_rows=("label", "size"),
                n_episodes=("global_episode_key", "nunique"),
                invalidity_prevalence=("label", "mean"),
            )
        )

        observed_role_tracks = set(
            role_summary["track"].tolist()
        )

        missing_tracks = sorted(
            set(expected_tracks) - observed_role_tracks
        )

        for track_name in expected_tracks:
            track_rows = role_summary.loc[
                role_summary["track"] == track_name
            ]

            if track_rows.empty:
                coverage_rows.append(
                    {
                        "seed": int(seed_value),
                        "split_role": split_role,
                        "track": track_name,
                        "n_rows": 0,
                        "n_episodes": 0,
                        "invalidity_prevalence": None,
                        "track_present": False,
                    }
                )
            else:
                summary_row = track_rows.iloc[0]

                coverage_rows.append(
                    {
                        "seed": int(seed_value),
                        "split_role": split_role,
                        "track": track_name,
                        "n_rows": int(summary_row["n_rows"]),
                        "n_episodes": int(
                            summary_row["n_episodes"]
                        ),
                        "invalidity_prevalence": float(
                            summary_row["invalidity_prevalence"]
                        ),
                        "track_present": True,
                    }
                )

coverage_summary = pd.DataFrame(coverage_rows)

print(
    coverage_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

coverage_by_role = (
    coverage_summary.groupby(
        ["split_role", "track"],
        as_index=False,
    )
    .agg(
        seeds_present=("track_present", "sum"),
        seeds_total=("track_present", "size"),
        min_n_episodes=("n_episodes", "min"),
        max_n_episodes=("n_episodes", "max"),
        min_n_rows=("n_rows", "min"),
        max_n_rows=("n_rows", "max"),
    )
)

print_header("TRACK COVERAGE SUMMARY ACROSS SEEDS")

print(
    coverage_by_role.to_string(
        index=False,
    )
)

calibration_coverage = coverage_summary.loc[
    coverage_summary["split_role"] == "calibration"
].copy()

evaluation_coverage = coverage_summary.loc[
    coverage_summary["split_role"] == "evaluation"
].copy()

assertions = {
    "all_expected_tracks_observed_in_source": (
        len(expected_tracks) == 4
    ),
    "all_calibration_track_rows_assigned": bool(
        calibration_coverage["track_present"].all()
    ),
    "all_evaluation_track_rows_assigned": bool(
        evaluation_coverage["track_present"].all()
    ),
    "calibrator_fitted": False,
    "files_modified": False,
}

print_header("TRACK-COVERAGE ASSERTIONS")

print(
    json.dumps(
        assertions,
        indent=2,
        ensure_ascii=False,
    )
)

if assertions["calibrator_fitted"]:
    raise AssertionError(
        "A calibrator must not be fitted in this audit."
    )

if assertions["files_modified"]:
    raise AssertionError(
        "A file must not be modified in this audit."
    )

print_header(
    "TRACK-COVERAGE AUDIT COMPLETE — NO FILES WERE MODIFIED"
)

In [ ]:
from pathlib import Path
import subprocess
import json
from collections import Counter

REPOSITORY_PATH = Path("/content/cognitive-self-model").resolve()

if not REPOSITORY_PATH.exists():
    raise FileNotFoundError(
        f"Repository directory does not exist: {REPOSITORY_PATH}"
    )

def run_command(command: list[str]) -> str:
    result = subprocess.run(
        command,
        cwd=REPOSITORY_PATH,
        text=True,
        capture_output=True,
        check=False,
    )
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(command)}\n\n{output}"
        )
    return output

git_snapshot = {
    "repository_path": str(REPOSITORY_PATH),
    "git_status_short": run_command(["git", "status", "--short"]),
    "current_branch": run_command(["git", "branch", "--show-current"]),
    "remotes": run_command(["git", "remote", "-v"]),
    "recent_commits": run_command(["git", "log", "--oneline", "-10"]),
}

excluded_directory_names = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".venv",
    "venv",
    "node_modules",
}

repository_files = sorted(
    path.relative_to(REPOSITORY_PATH).as_posix()
    for path in REPOSITORY_PATH.rglob("*")
    if path.is_file()
    and not any(part in excluded_directory_names for part in path.parts)
)

extension_counts = Counter(
    Path(file_path).suffix.lower() or "<no_extension>"
    for file_path in repository_files
)

guidance_candidates = [
    "README.md",
    "AGENTS.md",
    "AI_PROJECT_INSTRUCTIONS.md",
    "CONTRIBUTING.md",
    "pyproject.toml",
    "requirements.txt",
]

guidance_files_present = [
    file_name
    for file_name in guidance_candidates
    if (REPOSITORY_PATH / file_name).is_file()
]

inventory = {
    "git_snapshot": git_snapshot,
    "repository_file_count": len(repository_files),
    "extension_counts": dict(sorted(extension_counts.items())),
    "guidance_files_present": guidance_files_present,
    "repository_files": repository_files,
}

print("=" * 100)
print("REPOSITORY CONTROL PLANE — READ-ONLY INVENTORY")
print("=" * 100)
print(json.dumps(git_snapshot, indent=2, ensure_ascii=False))

print("\n" + "=" * 100)
print(f"TRACKED WORKING FILE INVENTORY: {len(repository_files)} files")
print("=" * 100)

for index, file_path in enumerate(repository_files, start=1):
    print(f"{index:03d}. {file_path}")

print("\n" + "=" * 100)
print("FILE TYPE SUMMARY")
print("=" * 100)
for extension, count in sorted(extension_counts.items()):
    print(f"{extension}: {count}")

print("\n" + "=" * 100)
print("GUIDANCE FILES PRESENT")
print("=" * 100)
for file_name in guidance_files_present:
    print(file_name)

output_path = REPOSITORY_PATH / "control_plane_inventory_preview.json"
output_path.write_text(
    json.dumps(inventory, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("\n" + "=" * 100)
print("READ-ONLY INSPECTION COMPLETE")
print("NOTE: Only a local preview file was created:")
print(output_path.name)
print("Do not stage or commit it.")
print("=" * 100)

In [ ]:
from pathlib import Path
import subprocess
import os

def is_git_repository(path: Path) -> bool:
    result = subprocess.run(
        ["git", "-C", str(path), "rev-parse", "--is-inside-work-tree"],
        text=True,
        capture_output=True,
        check=False,
    )
    return result.returncode == 0 and result.stdout.strip() == "true"

search_roots = [
    Path.cwd(),
    Path("/content"),
    Path("/workspace"),
    Path("/home"),
]

candidate_paths = set()

for root in search_roots:
    if not root.exists():
        continue

    # خود root و حداکثر دو سطح فرزند را بررسی می‌کنیم.
    candidate_paths.add(root)

    try:
        for child in root.iterdir():
            if child.is_dir():
                candidate_paths.add(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidate_paths.add(grandchild)
    except PermissionError:
        pass

git_repositories = sorted(
    str(path.resolve())
    for path in candidate_paths
    if is_git_repository(path)
)

print("=" * 100)
print("GIT REPOSITORY DISCOVERY — READ ONLY")
print("=" * 100)
print(f"Current working directory: {Path.cwd().resolve()}")
print("\nDetected Git repositories:")

if git_repositories:
    for index, repository_path in enumerate(git_repositories, start=1):
        print(f"{index}. {repository_path}")
else:
    print("No Git repository was found in the checked paths.")
    print("\nChecked roots:")
    for root in search_roots:
        print(f"- {root}")

print("=" * 100)

In [ ]:
from pathlib import Path
from collections import Counter
import json
import subprocess

REPOSITORY_PATH = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

def run_command(command: list[str]) -> str:
    result = subprocess.run(
        command,
        cwd=REPOSITORY_PATH,
        text=True,
        capture_output=True,
        check=False,
    )

    output = (result.stdout + result.stderr).strip()

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(command)}\n\n{output}"
        )

    return output

git_top_level = Path(
    run_command(["git", "rev-parse", "--show-toplevel"])
).resolve()

if git_top_level != REPOSITORY_PATH:
    raise RuntimeError(
        "REPOSITORY_PATH is not the Git repository root.\n"
        f"Configured path: {REPOSITORY_PATH}\n"
        f"Git top-level:    {git_top_level}"
    )

excluded_directory_names = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".venv",
    "venv",
    "node_modules",
}

repository_files = sorted(
    path.relative_to(REPOSITORY_PATH).as_posix()
    for path in REPOSITORY_PATH.rglob("*")
    if path.is_file()
    and not any(part in excluded_directory_names for part in path.parts)
)

extension_counts = Counter(
    Path(file_path).suffix.lower() or "<no_extension>"
    for file_path in repository_files
)

guidance_candidates = [
    "README.md",
    "AGENTS.md",
    "AI_PROJECT_INSTRUCTIONS.md",
    "CONTRIBUTING.md",
    "pyproject.toml",
    "requirements.txt",
]

guidance_files_present = [
    file_name
    for file_name in guidance_candidates
    if (REPOSITORY_PATH / file_name).is_file()
]

git_snapshot = {
    "repository_path": str(REPOSITORY_PATH),
    "git_top_level": str(git_top_level),
    "git_status_short": run_command(["git", "status", "--short"]),
    "current_branch": run_command(["git", "branch", "--show-current"]),
    "remotes": run_command(["git", "remote", "-v"]),
    "recent_commits": run_command(["git", "log", "--oneline", "-10"]),
}

print("=" * 100)
print("REPOSITORY CONTROL PLANE — READ-ONLY INVENTORY")
print("=" * 100)
print(json.dumps(git_snapshot, indent=2, ensure_ascii=False))

print("\n" + "=" * 100)
print(f"WORKING-TREE FILE INVENTORY: {len(repository_files)} files")
print("=" * 100)

for index, file_path in enumerate(repository_files, start=1):
    print(f"{index:03d}. {file_path}")

print("\n" + "=" * 100)
print("FILE TYPE SUMMARY")
print("=" * 100)

for extension, count in sorted(extension_counts.items()):
    print(f"{extension}: {count}")

print("\n" + "=" * 100)
print("GUIDANCE / PROJECT ENTRY FILES PRESENT")
print("=" * 100)

if guidance_files_present:
    for file_name in guidance_files_present:
        print(f"- {file_name}")
else:
    print("- None found among the expected top-level files.")

print("\n" + "=" * 100)
print("COMPLETE — NO FILES WERE CREATED, MODIFIED, MOVED, OR DELETED")
print("=" * 100)

In [ ]:
from pathlib import Path

REPOSITORY_PATH = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

files_to_read = [
    "README.md",
    "AGENTS.md",
    "AI_PROJECT_INSTRUCTIONS.md",
    "CONTRIBUTING.md",
    "pyproject.toml",
    "requirements.txt",
]

for relative_path in files_to_read:
    file_path = REPOSITORY_PATH / relative_path

    if not file_path.is_file():
        continue

    print("\n" + "=" * 100)
    print(relative_path)
    print("=" * 100)
    print(file_path.read_text(encoding="utf-8", errors="replace")[:30000])

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import subprocess

REPOSITORY_PATH = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

def run_command(command: list[str]) -> str:
    result = subprocess.run(
        command,
        cwd=REPOSITORY_PATH,
        text=True,
        capture_output=True,
        check=False,
    )
    output = (result.stdout + result.stderr).strip()

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(command)}\n\n{output}"
        )
    return output

git_top_level = Path(
    run_command(["git", "rev-parse", "--show-toplevel"])
).resolve()

if git_top_level != REPOSITORY_PATH:
    raise RuntimeError(
        f"Repository root mismatch.\n"
        f"Configured: {REPOSITORY_PATH}\n"
        f"Git root:   {git_top_level}"
    )

tracked_files = sorted(
    line.strip()
    for line in run_command(["git", "ls-files"]).splitlines()
    if line.strip()
)

status_porcelain_v1 = run_command(["git", "status", "--porcelain=v1"])
changed_or_untracked_paths = [
    line for line in status_porcelain_v1.splitlines() if line.strip()
]

files_by_top_level_directory = defaultdict(list)
extension_counts = Counter()

for relative_path in tracked_files:
    path = Path(relative_path)
    top_level = path.parts[0] if len(path.parts) > 1 else "<repository_root>"
    files_by_top_level_directory[top_level].append(relative_path)
    extension_counts[path.suffix.lower() or "<no_extension>"] += 1

git_snapshot = {
    "repository_path": str(REPOSITORY_PATH),
    "current_branch": run_command(["git", "branch", "--show-current"]),
    "head_commit": run_command(["git", "log", "-1", "--format=%H"]),
    "head_commit_summary": run_command(["git", "log", "-1", "--format=%h %s"]),
    "remotes": run_command(["git", "remote", "-v"]),
    "working_tree_status": (
        "CLEAN" if not changed_or_untracked_paths else "HAS_CHANGES_OR_UNTRACKED_FILES"
    ),
    "recent_commits": run_command(["git", "log", "--oneline", "-10"]),
}

print("=" * 100)
print("GIT SNAPSHOT — READ ONLY")
print("=" * 100)
print(json.dumps(git_snapshot, indent=2, ensure_ascii=False))

print("\n" + "=" * 100)
print("WORKING TREE DETAILS")
print("=" * 100)
if changed_or_untracked_paths:
    print("\n".join(changed_or_untracked_paths))
else:
    print("Working tree is clean.")

print("\n" + "=" * 100)
print(f"TRACKED FILE INVENTORY — {len(tracked_files)} files")
print("=" * 100)

for directory_name in sorted(files_by_top_level_directory):
    directory_files = files_by_top_level_directory[directory_name]
    print(f"\n[{directory_name}] — {len(directory_files)} files")
    for relative_path in directory_files:
        print(f"  - {relative_path}")

print("\n" + "=" * 100)
print("TRACKED FILE TYPE SUMMARY")
print("=" * 100)
for extension, count in sorted(extension_counts.items()):
    print(f"{extension}: {count}")

print("\n" + "=" * 100)
print("COMPLETE — NO FILES WERE CREATED, MODIFIED, MOVED, OR DELETED")
print("=" * 100)

In [ ]:
from pathlib import Path

REPOSITORY_PATH = Path(
    "/content/cognitive-self-model-scientific-structure"
).resolve()

text_extensions = {
    ".py", ".md", ".txt", ".json", ".yaml", ".yml", ".toml", ".csv"
}

priority_directories = [
    "configs",
    "docs",
    "reports",
    "src",
    "tests",
    "scripts",
]

max_characters_per_file = 12000

for directory_name in priority_directories:
    directory_path = REPOSITORY_PATH / directory_name

    if not directory_path.is_dir():
        continue

    candidate_files = sorted(
        path
        for path in directory_path.rglob("*")
        if path.is_file() and path.suffix.lower() in text_extensions
    )

    print("\n" + "=" * 100)
    print(f"{directory_name}/ — {len(candidate_files)} readable text files")
    print("=" * 100)

    for file_path in candidate_files:
        relative_path = file_path.relative_to(REPOSITORY_PATH).as_posix()
        content = file_path.read_text(encoding="utf-8", errors="replace")

        print("\n" + "-" * 100)
        print(relative_path)
        print("-" * 100)
        print(content[:max_characters_per_file])

        if len(content) > max_characters_per_file:
            print(
                f"\n[Preview truncated after {max_characters_per_file} characters; "
                f"file length: {len(content)} characters.]"
            )